# ARC APEX-5: Duck-agent (Qwen3.8-27B FP8) + NVARC 4B residual floor


REPL program-synthesis agent (python + submit tools, verified on the demos) on the hidden test set. Config:

```
{
 "model_glob": "/kaggle/input/**/qwen3-8-27b-fp8-repacked-v1/**/config.json",
 "overlay": "/kaggle/input/**/taaf-duck-qwen38-serving-v1/serving_wheels/*.whl",
 "parsers": [
  "--tool-call-parser",
  "qwen3_coder",
  "--reasoning-parser",
  "qwen3"
 ],
 "server_ladder": [
  [
   "--reasoning-config",
   "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}",
   "--kv-cache-dtype",
   "fp8",
   "--language-model-only",
   "--speculative-config",
   "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"
  ],
  [
   "--reasoning-config",
   "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}",
   "--kv-cache-dtype",
   "fp8",
   "--speculative-config",
   "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"
  ],
  [
   "--reasoning-config",
   "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}",
   "--kv-cache-dtype",
   "fp8",
   "--speculative-config",
   "{\"method\": \"mtp\", \"num_speculative_tokens\": 2}"
  ],
  [
   "--reasoning-config",
   "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}",
   "--kv-cache-dtype",
   "fp8"
  ],
  [
   "--reasoning-config",
   "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}"
  ],
  []
 ],
 "max_model_len": 81920,
 "max_num_seqs": 16,
 "mode": "direct",
 "samples": 2,
 "concurrency": 10,
 "min_start_s": 600,
 "template_kwargs": {
  "reasoning_effort": "xhigh"
 },
 "temperature": 1.0,
 "top_p": 0.95,
 "top_k": 20,
 "thinking_token_budget": 40960,
 "max_tokens": 49152,
 "rescue_tokens": 8192,
 "tok_per_s": 140
}
```


## APEX-5 delta (hybrid)
- Agent phase: this notebook verbatim (REPL/direct agent, server ladder, fallback-first).
- Agent deadline pulled in by 100 min. After submit_main kills the server, a residual
  phase runs the proven NVARC 4B TTT floor (crc32-stable seeds) on ONLY the tasks the
  agent left as fallback copies, and fills those slots (never displaces, never raises).
- Floor modules are byte-verbatim from the proven v8 triple-pass stack.


In [1]:
# Runs first and cannot fail the notebook: a valid submission exists before anything heavy.
import json, glob, time, os
from pathlib import Path
START = time.time()
Path("/kaggle/working/start_epoch.txt").write_text(str(START))
try:
    ch = json.load(open(glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)[0]))
    with open("/kaggle/working/submission.json", "w") as f:
        json.dump({t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]]
                   for t, task in ch.items()}, f)
    print("fallback submission written for", len(ch), "tasks; rerun =",
          os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
except Exception as e:
    print("fallback FAILED:", repr(e))
os.makedirs("/kaggle/working/agent", exist_ok=True)


fallback submission written for 240 tasks; rerun = None


In [2]:
%%writefile /kaggle/working/agent/grids.py
"""Grid <-> text, task prompts and failure feedback for the program-synthesis agent.

Grids are shown as rows of digits with no separators, the same layout NVARC's
formatter uses, plus explicit dimensions -- reasoning models miscount rows and
columns of long digit strings otherwise.
"""

SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles by writing Python programs. "
    "Each puzzle has a few input->output example pairs that share one hidden "
    "transformation rule. Grids are 2D lists of integers 0-9 (colours); 0 is "
    "usually the background."
)

INSTRUCTIONS = """Find the transformation rule that maps EVERY example input to its output, then write it as a Python function.

Requirements:
- Define `def transform(grid: list[list[int]]) -> list[list[int]]:`.
- Use only the Python standard library and numpy (already importable as `np` if you `import numpy as np`).
- The function must work for every example input AND the test input(s); do not hard-code example outputs.
- Return a new 2D list of ints 0-9 (numpy arrays are accepted).

Reason about the rule first, then give the final program in ONE ```python code block."""


def grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in g)


def shape(g):
    return (len(g), len(g[0]) if g else 0)


def show(g, name):
    r, c = shape(g)
    return f"{name} ({r}x{c}):\n{grid_text(g)}"


def task_prompt(task):
    parts = [INSTRUCTIONS, ""]
    for i, pair in enumerate(task["train"], 1):
        parts.append(f"## Example {i}")
        parts.append(show(pair["input"], "Input"))
        parts.append(show(pair["output"], "Output"))
        parts.append("")
    for j, t in enumerate(task["test"], 1):
        parts.append(f"## Test input {j}")
        parts.append(show(t["input"], "Input"))
        parts.append("")
    return "\n".join(parts).rstrip()


def diff_text(expected, got, max_cells=900):
    """Expected grid with wrong cells marked, so the model sees WHERE it failed.

    '.' marks a correct cell; a wrong cell shows `expected/got` inline is too
    wide, so two aligned grids are given: the expected output, and the
    program's output with correct cells replaced by '.'.
    """
    re_, ce = shape(expected)
    rg, cg = shape(got)
    if (re_, ce) != (rg, cg):
        return (f"Wrong output SIZE: expected {re_}x{ce}, your program returned {rg}x{cg}.\n"
                + show(expected, "Expected") + "\n" + show(got, "Your output"))
    if re_ * ce > max_cells:
        wrong = sum(expected[r][c] != got[r][c] for r in range(re_) for c in range(ce))
        return f"{wrong} of {re_ * ce} cells are wrong (grid too large to print the diff)."
    wrong = 0
    rows = []
    for r in range(re_):
        s = []
        for c in range(ce):
            if expected[r][c] == got[r][c]:
                s.append(".")
            else:
                s.append(str(int(got[r][c])))
                wrong += 1
        rows.append("".join(s))
    return (f"{wrong} of {re_ * ce} cells are wrong.\n"
            + show(expected, "Expected") + "\n"
            + f"Your output, with CORRECT cells shown as '.' ({re_}x{ce}):\n" + "\n".join(rows))


def feedback_prompt(results, train, max_examples=3):
    """Tell the model which examples failed and how; ask for a corrected program."""
    lines = []
    n_ok = sum(r["ok"] for r in results)
    lines.append(f"Your program reproduced {n_ok} of {len(results)} examples.")
    shown = 0
    for i, (r, pair) in enumerate(zip(results, train), 1):
        if r["ok"]:
            continue
        if shown >= max_examples:
            break
        shown += 1
        lines.append(f"\n## Example {i} failed")
        if r.get("error"):
            lines.append(f"Error: {r['error']}")
        else:
            lines.append(diff_text(pair["output"], r["output"]))
    lines.append("\nRe-examine the rule against ALL examples, then give the corrected "
                 "complete program in ONE ```python code block.")
    return "\n".join(lines)


Writing /kaggle/working/agent/grids.py


In [3]:
%%writefile /kaggle/working/agent/sandbox.py
"""Run model-written `transform` programs in isolated subprocesses.

Each job is (code, inputs). The program runs in a fresh interpreter with a
wall-clock timeout (and, on Linux, an address-space cap), so an infinite loop,
a crash or a memory blow-up costs one job, never the run. Anything the program
prints is swallowed; the result travels back as one marked JSON line.

Per input the result is {"output": grid} or {"error": message}. An output is
only accepted if it is a 2D rectangular grid of ints 0-9, 1..30 on each side.
"""
import json
import os
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

MARK = "@@ARC_RESULT@@"

RUNNER = r'''
import sys, json, io, contextlib, copy, traceback
MARK = "@@ARC_RESULT@@"
job = json.loads(sys.stdin.read())

def clean(res):
    try:
        import numpy as np
        if isinstance(res, np.ndarray):
            res = res.tolist()
    except Exception:
        pass
    if isinstance(res, tuple):
        res = list(res)
    if not isinstance(res, list) or not res:
        return None, "output is not a non-empty list of rows"
    rows = []
    for row in res:
        if hasattr(row, "tolist"):
            row = row.tolist()
        if not isinstance(row, (list, tuple)) or not row:
            return None, "output rows must be non-empty lists"
        out = []
        for v in row:
            try:
                iv = int(v)
            except Exception:
                return None, f"non-integer cell {v!r}"
            if iv != v or not 0 <= iv <= 9:
                return None, f"cell value {v!r} outside 0-9"
            out.append(iv)
        rows.append(out)
    w = len(rows[0])
    if any(len(r) != w for r in rows):
        return None, "output is not rectangular"
    if not (1 <= len(rows) <= 30 and 1 <= w <= 30):
        return None, f"output size {len(rows)}x{w} outside 1..30"
    return rows, None

results = []
sink = io.StringIO()
ns = {"__name__": "__arc_program__"}
try:
    with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
        exec(compile(job["code"], "<program>", "exec"), ns)
    fn = ns.get("transform")
    if not callable(fn):
        raise NameError("no function named transform")
except Exception as e:
    msg = "".join(traceback.format_exception_only(type(e), e)).strip()
    results = [{"error": "program failed to load: " + msg[-400:]} for _ in job["inputs"]]
else:
    for grid in job["inputs"]:
        try:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                res = fn(copy.deepcopy(grid))
            rows, err = clean(res)
            results.append({"output": rows} if err is None else {"error": err})
        except Exception as e:
            tb = traceback.format_exc().strip().splitlines()
            results.append({"error": " | ".join(tb[-3:])[-400:]})
sys.stdout.write(MARK + json.dumps(results) + "\n")
sys.stdout.flush()
'''


def _limits(mem_bytes):
    def apply():
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except Exception:
            pass
    return apply if os.name == "posix" else None


def run_one(code, inputs, timeout=10.0, mem_bytes=2 << 30):
    """Run one program on a list of input grids; never raises."""
    try:
        p = subprocess.run(
            [sys.executable, "-c", RUNNER],
            input=json.dumps({"code": code, "inputs": inputs}),
            capture_output=True, text=True, timeout=timeout,
            preexec_fn=_limits(mem_bytes),
            env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                     MKL_NUM_THREADS="1", PYTHONHASHSEED="0"))
    except subprocess.TimeoutExpired:
        return [{"error": f"timed out after {timeout:.0f}s"} for _ in inputs]
    for line in reversed(p.stdout.splitlines()):
        if line.startswith(MARK):
            try:
                res = json.loads(line[len(MARK):])
                if len(res) == len(inputs):
                    return res
            except Exception:
                break
    tail = (p.stderr or "").strip().splitlines()[-2:]
    why = " | ".join(tail)[-300:] or f"exit code {p.returncode}"
    return [{"error": "program crashed: " + why} for _ in inputs]


def run_many(jobs, timeout=10.0, workers=None, mem_bytes=2 << 30):
    """jobs: list of (code, inputs). Returns results in the same order."""
    workers = workers or max(1, (os.cpu_count() or 2) - 1)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(run_one, c, i, timeout, mem_bytes) for c, i in jobs]
        return [f.result() for f in futs]


Writing /kaggle/working/agent/sandbox.py


In [4]:
%%writefile /kaggle/working/agent/backend.py
"""LLM backends. The agent only needs one call: complete a batch of chats.

`VLLMBackend` runs offline on the Kaggle GPUs. `MockBackend` replays scripted
programs so the whole harness can be exercised on CPU with no model.
"""
import re
import time

CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)
THINK_RE = re.compile(r"<think>.*?</think>", re.S)


def extract_code(text):
    """The last fenced block that defines transform(); falls back to any block
    containing it, then to a bare `def transform` in unfenced text."""
    if not text:
        return None
    body = THINK_RE.sub("", text)
    if "<think>" in body and "</think>" not in body:
        return None                      # truncated inside the reasoning
    blocks = [b for b in CODE_RE.findall(body) if "def transform" in b]
    if blocks:
        return blocks[-1].strip()
    i = body.rfind("def transform")
    if i >= 0:                           # unfenced: take the function to the end
        return "import numpy as np\n" + body[i:].strip()
    return None


class VLLMBackend:
    """Offline vLLM chat. One engine over all GPUs (tensor parallel)."""

    def __init__(self, model, tp=4, max_model_len=32768, gpu_mem=0.90,
                 max_tokens=16384, temperature=0.8, top_p=0.95, seed=0,
                 chat_kwargs=None, quantization=None, enforce_eager=False):
        from vllm import LLM, SamplingParams
        self.SamplingParams = SamplingParams
        self.llm = LLM(model=model, tensor_parallel_size=tp, max_model_len=max_model_len,
                       gpu_memory_utilization=gpu_mem, seed=seed, trust_remote_code=True,
                       quantization=quantization, enforce_eager=enforce_eager)
        self.max_tokens, self.temperature, self.top_p = max_tokens, temperature, top_p
        self.chat_kwargs = chat_kwargs or {}
        self.seed = seed
        self.tokenizer = self.llm.get_tokenizer()

    def prompt_tokens(self, messages):
        ids = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                                 **self.chat_kwargs)
        return len(ids)

    def chat(self, conversations, max_tokens=None, metas=None):
        """Returns [(text, n_generated_tokens, finish_reason)] in order."""
        params = [self.SamplingParams(temperature=self.temperature, top_p=self.top_p,
                                      max_tokens=max_tokens or self.max_tokens,
                                      seed=self.seed + i)
                  for i in range(len(conversations))]
        self.seed += len(conversations)
        outs = self.llm.chat(conversations, params, use_tqdm=False,
                             chat_template_kwargs=self.chat_kwargs)
        res = []
        for o in outs:
            c = o.outputs[0]
            res.append((c.text, len(c.token_ids), c.finish_reason))
        return res


class MockBackend:
    """Replays `script(task_id, round, attempt, conversation) -> text`.

    `metas` (task_id, attempt) travel beside the conversations, never inside
    them, so the vLLM path sees plain chat messages.
    """

    def __init__(self, script, delay=0.0):
        self.script = script
        self.delay = delay
        self.calls = 0

    def prompt_tokens(self, messages):
        return sum(len(m["content"]) for m in messages) // 3

    def chat(self, conversations, max_tokens=None, metas=None):
        self.calls += 1
        time.sleep(self.delay)
        out = []
        for conv, meta in zip(conversations, metas or [{}] * len(conversations)):
            rnd = sum(m["role"] == "assistant" for m in conv)
            text = self.script(meta.get("task_id"), rnd, meta.get("attempt", 0), conv)
            out.append((text, len(text) // 3, "stop"))
        return out


Writing /kaggle/working/agent/backend.py


In [5]:
%%writefile /kaggle/working/agent/agent.py
"""Batched program-synthesis agent for ARC tasks.

Every task gets K independent attempts (conversations). Each ROUND sends every
live conversation to the model in ONE batch -- that is what keeps the GPUs
busy -- then runs each extracted `transform` on the task's train and test inputs
in the sandbox. An attempt whose program reproduces every train output is
VERIFIED and stops. Otherwise the model is shown where its program failed and
asked again, up to R rounds.

The verification against the demos is the point of the whole approach: a
program that reproduces all examples is a far stronger selector than any
likelihood score (exp06: the NLL scorer prefers a wrong grid over the injected
truth for 83 % of missed outputs).

Answers per test output are voted over verified programs; if fewer than two
distinct verified answers exist, the gap is filled from the best unverified
programs (most train examples right), and the source of each answer is kept so
a fallback solver can be preferred over weak guesses.
"""
import time
from collections import Counter, defaultdict

from grids import SYSTEM, task_prompt, feedback_prompt
from backend import extract_code
from sandbox import run_many


def visible_reply(text, keep=6000):
    """What goes back into the conversation: the answer without the reasoning.

    Qwen3-style models wrap reasoning in <think>; gpt-oss via raw harmony output
    puts the answer after a 'final' channel marker. Either way the program is in
    the tail, and re-sending long reasoning would crowd out the context.
    """
    t = text or ""
    if "</think>" in t:
        t = t.split("</think>")[-1]
    for marker in ("<|channel|>final<|message|>", "assistantfinal"):
        if marker in t:
            t = t.split(marker)[-1]
    t = t.strip()
    return t[-keep:] if len(t) > keep else t


def _key(g):
    return tuple(tuple(r) for r in g)


class Attempt:
    __slots__ = ("task_id", "idx", "messages", "rounds", "status", "programs",
                 "tokens", "note")

    def __init__(self, task_id, idx, prompt):
        self.task_id, self.idx = task_id, idx
        self.messages = [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": prompt}]
        self.rounds = 0
        self.status = "active"          # active | verified | failed | dropped
        self.programs = []              # dicts: round, code, n_ok, n_train, test
        self.tokens = 0
        self.note = ""


def solve(tasks, backend, samples=4, rounds=3, max_tokens=None, deadline=None,
          stop_after=2, max_prompt_tokens=None, sandbox_timeout=10.0,
          sandbox_workers=None, log=print):
    """tasks: {task_id: {"train": [...], "test": [...]}}.

    Returns (answers, report). answers[task_id] is a list over test inputs of
    [(grid, source), ...] best first; report holds per-attempt diagnostics.
    """
    t0 = time.time()
    deadline = deadline or float("inf")
    attempts = [Attempt(tid, i, task_prompt(task))
                for tid, task in tasks.items() for i in range(samples)]
    verified_count = Counter()
    rounds_log = []

    for rnd in range(1, rounds + 1):
        live = [a for a in attempts if a.status == "active"
                and verified_count[a.task_id] < stop_after]
        for a in attempts:
            if a.status == "active" and verified_count[a.task_id] >= stop_after:
                a.status, a.note = "failed", "task already verified"
        if not live:
            break
        if time.time() > deadline:
            log(f"round {rnd}: deadline reached with {len(live)} attempts live")
            break
        if max_prompt_tokens:
            keep = []
            for a in live:
                if backend.prompt_tokens(a.messages) > max_prompt_tokens:
                    a.status, a.note = "dropped", "prompt too long"
                else:
                    keep.append(a)
            live = keep
            if not live:
                break

        tg = time.time()
        replies = backend.chat([a.messages for a in live], max_tokens=max_tokens,
                               metas=[{"task_id": a.task_id, "attempt": a.idx} for a in live])
        gen_s = time.time() - tg

        jobs, owners = [], []
        for a, (text, ntok, finish) in zip(live, replies):
            a.rounds += 1
            a.tokens += ntok
            code = extract_code(text)
            a.messages.append({"role": "assistant", "content": visible_reply(text)})
            if code is None:
                why = ("Your reply was cut off before the program." if finish == "length"
                       else "No ```python block defining transform() was found.")
                a.messages.append({"role": "user", "content":
                                   why + " Give the complete program in ONE ```python code block."})
                if a.rounds >= rounds:
                    a.status, a.note = "failed", "no code"
                continue
            task = tasks[a.task_id]
            inputs = [p["input"] for p in task["train"]] + [t["input"] for t in task["test"]]
            jobs.append((code, inputs))
            owners.append((a, code))

        ts = time.time()
        results = run_many(jobs, timeout=sandbox_timeout, workers=sandbox_workers)
        run_s = time.time() - ts

        n_ver = 0
        for (a, code), res in zip(owners, results):
            task = tasks[a.task_id]
            ntr = len(task["train"])
            train_res = []
            for r, pair in zip(res[:ntr], task["train"]):
                ok = "output" in r and r["output"] == pair["output"]
                train_res.append({"ok": ok, "output": r.get("output"), "error": r.get("error")})
            n_ok = sum(t["ok"] for t in train_res)
            test_out = [r.get("output") for r in res[ntr:]]
            a.programs.append({"round": a.rounds, "code": code, "n_ok": n_ok,
                               "n_train": ntr, "test": test_out,
                               "test_errors": [r.get("error") for r in res[ntr:]]})
            if n_ok == ntr and all(o is not None for o in test_out):
                a.status = "verified"
                verified_count[a.task_id] += 1
                n_ver += 1
            elif a.rounds >= rounds:
                a.status, a.note = "failed", f"best {max(p['n_ok'] for p in a.programs)}/{ntr}"
            else:
                a.messages.append({"role": "user",
                                   "content": feedback_prompt(train_res, task["train"])})

        toks = sum(r[1] for r in replies)
        rounds_log.append({"round": rnd, "live": len(live), "programs": len(jobs),
                           "verified_now": n_ver, "gen_s": gen_s, "run_s": run_s,
                           "tokens": toks, "tok_per_s": toks / max(gen_s, 1e-9)})
        log(f"round {rnd}: {len(live)} live, {len(jobs)} programs, {n_ver} newly verified, "
            f"tasks verified {sum(1 for t in tasks if verified_count[t])}/{len(tasks)}, "
            f"gen {gen_s:.0f}s ({toks / max(gen_s, 1e-9):.0f} tok/s) run {run_s:.1f}s")

    answers = {tid: aggregate(tid, tasks[tid], attempts) for tid in tasks}
    report = {
        "rounds": rounds_log,
        "wall_s": time.time() - t0,
        "attempts": [{"task_id": a.task_id, "idx": a.idx, "status": a.status,
                      "rounds": a.rounds, "tokens": a.tokens, "note": a.note,
                      "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
                      "n_train": len(tasks[a.task_id]["train"]),
                      "programs": a.programs} for a in attempts],
    }
    return answers, report


def aggregate(tid, task, attempts):
    """Per test input: candidate grids best first, each tagged with its source.

    Verified programs vote; unverified programs are ranked by train examples
    solved, then by how many programs agree.
    """
    mine = [a for a in attempts if a.task_id == tid]
    out = []
    for j in range(len(task["test"])):
        ver = Counter()
        unv = defaultdict(lambda: [0, 0])          # key -> [best n_ok, votes]
        grids = {}
        for a in mine:
            for p in a.programs:
                g = p["test"][j] if j < len(p["test"]) else None
                if g is None:
                    continue
                k = _key(g)
                grids[k] = g
                if p["n_ok"] == p["n_train"]:
                    ver[k] += 1
                else:
                    u = unv[k]
                    u[0] = max(u[0], p["n_ok"])
                    u[1] += 1
        ranked = [(grids[k], f"verified x{n}") for k, n in ver.most_common()]
        seen = {k for k in ver}
        rest = sorted((k for k in unv if k not in seen),
                      key=lambda k: (-unv[k][0], -unv[k][1]))
        ranked += [(grids[k], f"unverified {unv[k][0]}/{len(task['train'])}") for k in rest]
        out.append(ranked)
    return out


def to_submission(answers, tasks, fallback=None):
    """Kaggle format. `fallback(task_id, j)` supplies grids when the agent has
    fewer than two; otherwise the test input is echoed."""
    sub = {}
    for tid, task in tasks.items():
        rows = []
        for j, t in enumerate(task["test"]):
            cands = [g for g, _ in answers.get(tid, [[]] * len(task["test"]))[j]]
            if fallback and len(cands) < 2:
                for g in fallback(tid, j):
                    if g not in cands:
                        cands.append(g)
            while len(cands) < 2:
                cands.append(t["input"])
            rows.append({"attempt_1": cands[0], "attempt_2": cands[1]})
        sub[tid] = rows
    return sub


def score(sub, solutions):
    """ARC Prize scoring: a task scores the fraction of its test outputs matched
    by either attempt; the total is the sum over tasks (and / n as a percent)."""
    total, per = 0.0, {}
    for tid, sols in solutions.items():
        rows = sub.get(tid, [])
        hit = 0
        for j, sol in enumerate(sols):
            if j < len(rows) and sol in (rows[j]["attempt_1"], rows[j]["attempt_2"]):
                hit += 1
        per[tid] = hit / len(sols)
        total += per[tid]
    return total, 100.0 * total / max(len(solutions), 1), per


Writing /kaggle/working/agent/agent.py


In [6]:
%%writefile /kaggle/working/agent/repl.py
"""The `python` tool: run an exploratory snippet with the task preloaded.

Each call is a fresh subprocess (nothing persists between calls, as in the
Tufa duck harness), so a crash or an infinite loop costs one call. The task is
injected as numpy arrays plus a few ARC helpers; stdout is captured and capped.

Optional function retention (dfranzen's ARC-AGI-3 fork of the duck harness): with a
store, plain top-level functions a call defines are kept as SOURCE and replayed into
later calls; variables never persist. Old transcripts show why: 30-38% of all
function code the model wrote re-defined a function it had already written, and
19 of 20 ab3q36 submits re-typed functions first written in a python call.
"""
import json
import os
import subprocess
import sys

HELPERS = r'''
import numpy as np
from collections import Counter, defaultdict, deque
import itertools, math, functools, copy, re

def show(g):
    """Digit rows, one line per row."""
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))

def diff(a, b):
    """Cells where a and b differ, as (row, col, a_value, b_value); size mismatch -> string."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shape mismatch {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in zip(*np.nonzero(a != b))]

def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components. bg defaults to the most common colour.
    Returns dicts: color (or colors), cells [(r,c)], bbox (r0,c0,r1,c1), size, shape (cropped mask)."""
    g = np.asarray(g)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    H, W = g.shape
    seen = np.zeros_like(g, dtype=bool)
    nbrs = [(-1,0),(1,0),(0,-1),(0,1)] + ([(-1,-1),(-1,1),(1,-1),(1,1)] if diagonal else [])
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            q, cells = deque([(r, c)]), []
            seen[r, c] = True
            while q:
                y, x = q.popleft(); cells.append((y, x))
                for dy, dx in nbrs:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] and g[ny, nx] != bg \
                            and (multicolor or g[ny, nx] == col):
                        seen[ny, nx] = True; q.append((ny, nx))
            ys, xs = zip(*cells)
            r0, c0, r1, c1 = min(ys), min(xs), max(ys), max(xs)
            mask = np.zeros((r1 - r0 + 1, c1 - c0 + 1), dtype=int)
            for y, x in cells:
                mask[y - r0, x - c0] = g[y, x]
            d = {"cells": cells, "bbox": (r0, c0, r1, c1), "size": len(cells), "shape": mask}
            if multicolor:
                d["colors"] = sorted({int(g[y, x]) for y, x in cells})
            else:
                d["color"] = int(col)
            out.append(d)
    return out
'''

RUNNER = r'''
import sys, json, io, contextlib, traceback
job = json.loads(sys.stdin.read())
ns = {"__name__": "__arc_repl__"}
exec(job["helpers"], ns)
import numpy as np
ns["train_inputs"] = [np.array(g) for g in job["train_inputs"]]
ns["train_outputs"] = [np.array(g) for g in job["train_outputs"]]
ns["test_inputs"] = [np.array(g) for g in job["test_inputs"]]
ns["train"] = list(zip(ns["train_inputs"], ns["train_outputs"]))
keep = job.get("retained")                # None: retention off (each call starts fresh)
preloaded = set(ns)
restore_failed = []
if keep is not None:
    # Replay what earlier calls kept: their imports, then their function definitions.
    # Source only -- no variable or array is ever carried over.
    for src in keep["imports"]:
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            pass
    for name, src in keep["defs"].items():
        try:
            exec(compile(src, "<kept>", "exec"), ns)
        except Exception:
            restore_failed.append(name)
buf = io.StringIO()
try:
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        exec(compile(job["code"], "<snippet>", "exec"), ns)
        if "result" in ns and ns["result"] is not None:
            print(repr(ns["result"]))
except Exception:
    tb = traceback.format_exc().strip().splitlines()
    buf.write("\n" + "\n".join(tb[-4:]))
sys.stdout.write("@@REPL@@" + json.dumps(buf.getvalue()) + "\n")
if keep is not None:
    import ast, builtins, dis, types
    defs, imports, rejected = {}, [], {}
    try:
        tree = ast.parse(job["code"])
    except SyntaxError:
        tree = ast.Module(body=[], type_ignores=[])
    bound = set()
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)) and not getattr(node, "level", 0):
            src = ast.get_source_segment(job["code"], node)
            if src:
                imports.append(src)
                bound |= {(a.asname or a.name).split(".")[0] for a in node.names}
    new = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    available = (set(dir(builtins)) | preloaded | set(keep["defs"]) | bound
                 | {n.name for n in new} | set(keep.get("bound", [])))
    for node in new:
        fn = ns.get(node.name)
        # only a def that actually ran and still holds the name (not reassigned later)
        if not (isinstance(fn, types.FunctionType) and fn.__code__.co_filename == "<snippet>"
                and fn.__code__.co_firstlineno == node.lineno):
            continue
        reads, todo = set(), [fn.__code__]
        while todo:
            co = todo.pop()
            reads |= {i.argval for i in dis.get_instructions(co)
                      if i.opname in ("LOAD_GLOBAL", "LOAD_NAME")}
            todo += [c for c in co.co_consts if isinstance(c, types.CodeType)]
        missing = sorted(r for r in reads if isinstance(r, str) and r not in available)
        if missing:
            rejected[node.name] = missing
        else:
            defs[node.name] = ast.get_source_segment(job["code"], node)
    sys.stdout.write("@@KEPT@@" + json.dumps({"defs": defs, "imports": imports, "bound": sorted(bound),
                                             "rejected": rejected,
                                             "restore_failed": restore_failed}) + "\n")
'''

PREAMBLE_DOC = (
    "Preloaded in every `python` call (each call starts fresh; nothing persists): "
    "`np`, `train_inputs`, `train_outputs`, `test_inputs` (lists of numpy int arrays), "
    "`train` (list of (input, output) pairs), and helpers `show(g)` (digit rows), "
    "`diff(a, b)` (differing cells as (row, col, a, b)), "
    "`objects(g, bg=None, diagonal=False, multicolor=False)` (connected components with "
    "color, cells, bbox, size, shape). Print compact summaries, never whole large grids."
)


RETAIN_DOC = (
    "Variables do not persist between calls, but every function you define with a plain "
    "top-level `def` is kept: later `python` calls and `submit` can call it without "
    "redefining it. A kept function may use builtins, the preloaded names, imports and "
    "other kept functions; pass anything else in as an argument.")


def new_store():
    """Per-attempt record of what python calls have kept (source only)."""
    return {"defs": {}, "imports": [], "bound": []}


def run_snippet(code, task, timeout=30.0, max_chars=4000, mem_bytes=2 << 30, store=None):
    """Run one snippet. With `store` (from new_store()), functions defined by earlier calls
    are replayed first and the ones this call defines are added to the store; a line at
    the end of the output says what was kept."""
    job = {"code": code, "helpers": HELPERS,
           "train_inputs": [p["input"] for p in task["train"]],
           "train_outputs": [p["output"] for p in task["train"]],
           "test_inputs": [t["input"] for t in task["test"]]}
    if store is not None:
        job["retained"] = store
    pre = None
    if os.name == "posix":
        def pre():
            try:
                import resource
                resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
            except Exception:
                pass
    try:
        p = subprocess.run([sys.executable, "-c", RUNNER], input=json.dumps(job),
                           capture_output=True, text=True, timeout=timeout, preexec_fn=pre,
                           env=dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1"))
    except subprocess.TimeoutExpired:
        return f"[timed out after {timeout:.0f}s]"
    out, kept = None, None
    for line in reversed(p.stdout.splitlines()):
        if line.startswith("@@KEPT@@") and kept is None:
            kept = json.loads(line[len("@@KEPT@@"):])
        if line.startswith("@@REPL@@"):
            out = json.loads(line[len("@@REPL@@"):])
            break
    if out is None:
        out = "[snippet crashed] " + " | ".join((p.stderr or "").strip().splitlines()[-2:])
    out = out.strip() or "[no output -- print() what you want to see]"
    if len(out) > max_chars:
        out = out[:max_chars] + f"\n... [truncated {len(out) - max_chars} chars]"
    if store is not None and kept is not None:
        out += _absorb(store, kept)
    return out


def _absorb(store, kept):
    """Merge one call's kept definitions into the store; return the note for the model."""
    for name in kept["restore_failed"]:
        store["defs"].pop(name, None)
    for src in kept["imports"]:
        if src not in store["imports"]:
            store["imports"].append(src)
    store["bound"] = sorted(set(store["bound"]) | set(kept["bound"]))
    notes = []
    for name, src in kept["defs"].items():
        if store["defs"].get(name) != src:
            store["defs"].pop(name, None)            # re-insert: newest definition last
            store["defs"][name] = src
    if kept["defs"]:
        notes.append("kept for later calls: " + ", ".join(sorted(kept["defs"])))
    for name, missing in kept["rejected"].items():
        notes.append(f"not kept: {name} (uses {', '.join(missing[:4])} from this snippet -- "
                     "pass it as an argument to keep the function)")
    if kept["restore_failed"]:
        notes.append("dropped (failed to reload): " + ", ".join(kept["restore_failed"]))
    return ("\n[" + "; ".join(notes) + "]") if notes else ""


def with_kept(code, store):
    """A submitted program plus the kept functions (and imports) it relies on.

    Returns (full_source, names_added). The program's own definitions win; kept
    functions are added only if the program uses them, transitively, and transform()
    itself is taken from the store if the program does not define one."""
    import ast
    if not store or not store["defs"]:
        return code, []
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code, []
    defined = set()
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            defined.add(node.name)
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            defined |= {(a.asname or a.name).split(".")[0] for a in node.names}
        elif isinstance(node, ast.Assign):
            defined |= {t.id for t in node.targets if isinstance(t, ast.Name)}

    def used(src):
        try:
            return {n.id for n in ast.walk(ast.parse(src)) if isinstance(n, ast.Name)}
        except SyntaxError:
            return set()

    need = (used(code) & set(store["defs"])) - defined
    if "transform" not in defined and "transform" in store["defs"]:
        need.add("transform")
    todo = list(need)
    while todo:
        for n in (used(store["defs"][todo.pop()]) & set(store["defs"])) - defined - need:
            need.add(n)
            todo.append(n)
    if not need:
        return code, []
    # Kept functions were written against the python tool's namespace (np, objects(), ...),
    # so the helper preamble goes in first; the program's own definitions still win.
    parts = ([HELPERS.strip()] + list(store["imports"])
             + [src for n, src in store["defs"].items() if n in need])
    return "\n".join(parts) + "\n\n" + code, sorted(need)


Writing /kaggle/working/agent/repl.py


In [7]:
%%writefile /kaggle/working/agent/client.py
"""Minimal OpenAI-compatible chat client for a local vLLM server (stdlib only).

Returns a normalised dict: content, reasoning, tool_calls [{id, name, arguments}],
finish_reason, prompt_tokens, completion_tokens. If the server's tool parser
misses a call that the model wrote as qwen3_coder markup in its text, the call
is recovered from the markup, as the Tufa harness does.
"""
import json
import re
import urllib.error
import urllib.request

TOOL_BLOCK = re.compile(r"<tool_call>\s*<function=([\w\-]+)>(.*?)</function>\s*</tool_call>", re.S)
PARAM = re.compile(r"<parameter=([\w\-]+)>\n?(.*?)\n?</parameter>", re.S)


def recover_markup(text):
    calls = []
    for name, body in TOOL_BLOCK.findall(text or ""):
        args = {k: v for k, v in PARAM.findall(body)}
        calls.append({"id": f"markup-{len(calls) + 1}", "name": name, "arguments": args})
    return calls


def strip_markup(text):
    return TOOL_BLOCK.sub("", text or "").strip()


class ChatClient:
    def __init__(self, base_url, model, temperature=0.6, top_p=0.95, top_k=20,
                 max_tokens=16384, thinking=True, timeout=1800, reasoning_effort=None,
                 thinking_token_budget=None, template_kwargs=None):
        self.url = base_url.rstrip("/") + "/chat/completions"
        self.model = model
        # top_k=None leaves it to the server (gpt-oss wants no top-k).
        self.sampling = {k: v for k, v in {"temperature": temperature, "top_p": top_p,
                                           "top_k": top_k}.items() if v is not None}
        self.reasoning_effort = reasoning_effort
        # vLLM >= 0.19 forces </think> after this many reasoning tokens (needs the server
        # flag --reasoning-config). Qwen3.8 otherwise spends whole 16k turns thinking.
        self.thinking_token_budget = thinking_token_budget
        # Extra chat-template kwargs, e.g. Qwen3.8's reasoning_effort (xhigh default /
        # medium / low) -- the template itself turns it into a system instruction.
        self.template_kwargs = template_kwargs or {}
        self.max_tokens = max_tokens
        self.thinking = thinking
        self.timeout = timeout

    def chat(self, messages, tools=None, max_tokens=None, seed=None, timeout=None,
             tool_choice=None, template_kwargs=None):
        # template_kwargs: per-request overrides on top of the client's own, e.g. the
        # reasoning-effort ladder stepping one turn down to "medium".
        payload = {"model": self.model, "messages": messages,
                   "max_tokens": max_tokens or self.max_tokens,
                   "chat_template_kwargs": {"enable_thinking": self.thinking, **self.template_kwargs,
                                            **(template_kwargs or {})},
                   **self.sampling}
        if seed is not None:
            payload["seed"] = seed
        if self.reasoning_effort:          # gpt-oss: low / medium / high
            payload["reasoning_effort"] = self.reasoning_effort
        if self.thinking_token_budget:
            payload["thinking_token_budget"] = self.thinking_token_budget
        if tools:
            payload["tools"] = tools
            # A named tool forces that call (vLLM guided decoding); default lets it choose.
            payload["tool_choice"] = ({"type": "function", "function": {"name": tool_choice}}
                                      if tool_choice else "auto")
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout or self.timeout) as r:
                body = json.loads(r.read().decode())
            return normalise(body)
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:500]
            overflow = "maximum context length" in detail or "context length" in detail
            if (e.code == 400 and not overflow and "thinking_token_budget" in payload
                    and any(w in detail.lower() for w in ("thinking", "budget", "reasoning"))):
                # A server started without --reasoning-config (the last rungs of the startup
                # ladder) may refuse the budget: drop it for every later call, not fail them all.
                self.thinking_token_budget = None
                return self.chat(messages, tools, max_tokens, seed, timeout, tool_choice, template_kwargs)
            return {"error": f"HTTP {e.code}: {detail}", "context_overflow": overflow}
        except Exception as e:
            # probe24 lost every result to one uncaught socket TimeoutError.
            text = f"{type(e).__name__}: {e}"
            return {"error": text[:300], "context_overflow": False,
                    "timeout": isinstance(e, TimeoutError) or "timed out" in text}


def normalise(body):
    ch = body["choices"][0]
    msg = ch.get("message", {}) or {}
    content = msg.get("content") or ""
    reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""
    calls = []
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        try:
            args = json.loads(fn.get("arguments") or "{}")
        except json.JSONDecodeError:
            args = {"_raw": fn.get("arguments")}
        calls.append({"id": tc.get("id") or f"call-{len(calls) + 1}",
                      "name": fn.get("name"), "arguments": args})
    if not calls:
        calls = recover_markup(content) or recover_markup(reasoning)
        content = strip_markup(content)
    usage = body.get("usage") or {}
    return {"content": content, "reasoning": reasoning, "tool_calls": calls,
            "finish_reason": ch.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0)}


Writing /kaggle/working/agent/client.py


In [8]:
%%writefile /kaggle/working/agent/repl_agent.py
"""REPL tool agent for ARC-AGI-2 tasks, modelled on the Tufa Labs duck harness.

The model gets two tools:
  python(code)  explore the task: train/test grids are preloaded as numpy
                arrays with ARC helpers; stdout comes back, capped.
  submit(code)  a program defining transform(grid). It is run on every train
                input; the reply says which examples it reproduced and shows a
                cell diff for the rest. Reproducing ALL of them verifies the
                program, records its test outputs and ends the attempt.

Many attempts run concurrently against one vLLM OpenAI server, which batches
them. Old turns are evicted when the context budget is exceeded; the system
prompt and the task stay. Answers are aggregated exactly as in agent.py.

Optional, off by default, from dfranzen's ARC-AGI-3 fork of the duck harness (see
run_attempt): reasoning-effort ladder on cut turns, function retention, kept
reasoning, hysteresis trimming, repetition-loop guard, timeout-as-cut.
"""
import json
import threading
import time
import zlib
from concurrent.futures import ThreadPoolExecutor

from agent import aggregate
from backend import extract_code
from grids import SYSTEM, show, diff_text
from repl import run_snippet, PREAMBLE_DOC, RETAIN_DOC, new_store, with_kept
from sandbox import run_one

TOOLS = [
    {"type": "function", "function": {
        "name": "python",
        "description": "Run Python to inspect the task. " + PREAMBLE_DOC,
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Python code; print() what you need."}},
            "required": ["code"]}}},
    {"type": "function", "function": {
        "name": "submit",
        "description": ("Submit a complete program defining transform(grid) -> grid. It is run on "
                        "every example input; you are told which examples it reproduces. "
                        "When it reproduces all of them it is accepted and you are done."),
        "parameters": {"type": "object", "properties": {
            "code": {"type": "string", "description": "Full Python source defining transform(grid)."}},
            "required": ["code"]}}},
]

REPL_INSTRUCTIONS = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. Then call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy.

Submit EARLY and often: submitting is free, and a failing submission tells you exactly which examples and which cells are wrong -- that is the fastest way to find the rule. Do not wait for certainty. If a submission fails, study the diff and fix the rule rather than special-casing examples."""

# probe24: every failed attempt spent 30-73k tokens exploring and never submitted
# once, so it never saw a diff. After this many python calls without a submit,
# the tool output carries a reminder.
NUDGE_EVERY = 6
NUDGE = ("\n\n[harness] You have run {n} python calls without submitting. Submitting is free "
         "and returns a cell-level diff against every example. Submit your current best "
         "transform now, then refine it from the diff.")

# probe9q38: every failed attempt had 1-3 turns cut at 16k tokens mid-thought, while
# solves used 2-3k tokens a turn. The thinking-cap treatment lowers the per-turn cap
# and, when a turn is cut, orders the model to act instead of resuming the analysis.
THINKCAP_HINT = ("[harness] You hit the per-turn thinking limit. Stop analysing. In your next "
                 "reply, call `python` to test your current best hypothesis in code, or "
                 "`submit` a transform implementing it -- a failing submit shows exactly "
                 "which cells are wrong.")


# The probe24 wording, kept verbatim so a run with the nudge off matches it exactly.
REPL_INSTRUCTIONS_V1 = """Solve this ARC puzzle. The examples below share one hidden transformation rule.

Work like a scientist: use the `python` tool to measure the examples (objects, colours, sizes, positions, differences between input and output), form a hypothesis, and test it in code. When you have a program that you believe reproduces EVERY example, call `submit` with the full source defining `transform(grid)`. Use only the standard library and numpy. If a submission fails, study the diff and fix the rule rather than special-casing examples."""


# probe9q38 transcripts: failures parse and count grids IN THE REASONING, cell by cell,
# often filling a whole 16k-token turn before any tool call ("col 7=8, 8=0, 9=0 ...").
# Tufa's harness says the same thing to its model. Optional, its own arm.
IN_PYTHON_RULE = """

Never read, count, index or transcribe grid cells in your head. Do ALL of that with the `python` tool -- the grids are already loaded as numpy arrays. Keep your own reasoning to hypotheses and plans; let code do the looking."""


def repl_prompt(task, early_submit=True, in_python=False):
    head = REPL_INSTRUCTIONS if early_submit else REPL_INSTRUCTIONS_V1
    parts = [head + (IN_PYTHON_RULE if in_python else ""), ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    return "\n".join(parts).rstrip()


def verify(code, task, timeout=10.0):
    """Run a submitted program; return (n_ok, test_outputs, feedback_text)."""
    tr = task["train"]
    res = run_one(code, [p["input"] for p in tr] + [t["input"] for t in task["test"]],
                  timeout=timeout)
    n_ok, lines, shown = 0, [], 0
    for i, (r, p) in enumerate(zip(res[:len(tr)], tr), 1):
        if "output" in r and r["output"] == p["output"]:
            n_ok += 1
            continue
        if shown < 2:
            shown += 1
            lines.append(f"## Example {i} failed")
            lines.append(f"Error: {r['error']}" if "error" in r else diff_text(p["output"], r["output"]))
    test_out = [r.get("output") for r in res[len(tr):]]
    test_err = [r.get("error") for r in res[len(tr):] if "error" in r]
    if n_ok == len(tr):
        if test_err:
            return n_ok, test_out, ("Reproduces all examples but FAILS on a test input: "
                                    + test_err[0] + " -- fix it and submit again.")
        return n_ok, test_out, f"Accepted: reproduces all {len(tr)} examples."
    return n_ok, test_out, (f"Reproduced {n_ok} of {len(tr)} examples.\n" + "\n".join(lines)
                            + "\nFix the rule and submit again.")


def _est_tokens(messages):
    return len(json.dumps(messages)) // 3


def trim(messages, budget, keep_recent=6, drain=0, state=None):
    """Drop the oldest assistant/tool/user groups after the task prompt until the
    estimate fits. A group starts at an assistant message, so tool results are
    never separated from the call that produced them.

    With `drain` and a per-attempt `state` dict the cut has hysteresis (dfranzen's
    ARC-AGI-3 harness): it only moves when the estimate exceeds `budget`, and then
    drops down to `budget - drain`. Between those moments the retained history is
    byte-identical from turn to turn, so the server's prefix cache keeps serving it;
    without it, every turn past the budget shifts the cut by one group and
    re-prefills everything after the task prompt."""
    if drain and state is not None:
        return _trim_hysteresis(messages, budget, keep_recent, drain, state)
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    dropped = 0
    while _est_tokens(head + rest) > budget and len(starts) > keep_recent:
        cut = starts[1]
        rest = rest[cut:]
        starts = [i - cut for i in starts[1:]]
        dropped += 1
    if dropped:
        note = {"role": "user", "content": f"[{dropped} older turns were removed to save context.]"}
        rest = [note] + rest
    return head + rest


def _trim_hysteresis(messages, budget, keep_recent, drain, state):
    head, rest = messages[:2], messages[2:]
    starts = [i for i, m in enumerate(rest) if m["role"] == "assistant"]
    max_cut = max(0, len(starts) - keep_recent)

    def view(cut):
        if cut <= 0:
            return head + rest
        note = {"role": "user", "content": f"[{cut} older turns were removed to save context.]"}
        return head + [note] + rest[starts[cut]:]

    cut = min(state.get("cut", 0), max_cut)
    if _est_tokens(view(cut)) > budget:
        while cut < max_cut and _est_tokens(view(cut)) > budget - drain:
            cut += 1
        state["evictions"] = state.get("evictions", 0) + 1
    state["cut"] = cut
    return view(cut)


def degenerate(text, min_chars=500, ratio=0.01):
    """A repetition loop rather than text: zlib compresses it below `ratio`. dfranzen saw a
    reasoning-only reply of 6,144 '!' under speculative decoding; prose compresses to
    0.2-0.3, an ASCII grid to ~0.02, so 0.01 only catches genuine loops."""
    if not text or len(text) < min_chars:
        return False
    raw = text.encode("utf-8", "replace")
    return len(zlib.compress(raw, 6)) / len(raw) < ratio


DEGENERATE_STUB = ("(previous reply was a repetition loop and has been omitted; do not continue "
                   "that pattern)")


def make_tools(retain=False):
    """The two tools; with function retention their descriptions say what persists."""
    if not retain:
        return TOOLS
    py = dict(TOOLS[0]["function"], description=(
        "Run Python to inspect the task. "
        + PREAMBLE_DOC.replace(" (each call starts fresh; nothing persists)", "") + " " + RETAIN_DOC))
    sub = dict(TOOLS[1]["function"], description=(
        TOOLS[1]["function"]["description"] + " The program may call functions kept from your "
        "python calls (they are added automatically); if it defines no transform(), the kept "
        "transform() is submitted."))
    return [{"type": "function", "function": py}, {"type": "function", "function": sub}]


class _Shared:
    def __init__(self):
        self.lock = threading.Lock()
        self.verified = {}


class Attempt:
    def __init__(self, task_id, idx):
        self.task_id, self.idx = task_id, idx
        self.programs, self.status, self.note = [], "active", ""
        self.turns = self.python_calls = self.submits = 0
        self.python_at_submit = self.nudges = self.cuts = self.forced = 0
        self.gen_tokens = self.prompt_tokens = 0
        # ARC-AGI-3 takeaways (all zero unless their option is on)
        self.effort_steps = self.timeouts = self.degenerate = self.kept_used = 0
        self.evictions = 0
        self.wall = 0.0
        # Everything the model thought, ran and saw -- so a failure can be read as
        # "right idea, out of budget" vs "never found the rule".
        self.transcript = []


TIMEOUT_HINT = ("[harness] Your previous reply took too long and was abandoned. Keep your "
                "reasoning short: call `python` to test your current best hypothesis, or `submit`.")


def run_attempt(att, task, client, shared, deadline, max_turns=40, max_gen_tokens=120000,
                context_budget=28000, stop_after=1, seed=0, snippet_timeout=30.0,
                nudge_every=NUDGE_EVERY, turn_max_tokens=None, in_python=False,
                force_submit_after=0, effort_ladder=None, retain_functions=False,
                keep_reasoning=False, reasoning_key="reasoning_content", keep_recent=6,
                trim_drain=0, degenerate_guard=False, timeout_as_cut=False):
    """One attempt at one task. The options after force_submit_after are the ARC-AGI-3
    (dfranzen / duck harness) takeaways, each off by default so the baseline is unchanged:

    effort_ladder     e.g. ["medium", "low"]: after a turn is cut at max_tokens (or times
                      out, with timeout_as_cut) the next turns run one reasoning_effort rung
                      lower; a submit that reproduces more examples than before restores
                      the client's own effort.
    retain_functions  python-defined functions persist (as source) into later python calls
                      and into submits that use them.
    keep_reasoning    send each completed turn's reasoning back under `reasoning_key`
                      (pair with chat-template kwarg preserve_thinking=True). Cut turns and
                      repetition loops are never sent back.
    trim_drain        hysteresis for the context trim (see trim()); keep_recent groups are
                      never evicted.
    degenerate_guard  a repetition-loop reply is replaced by a stub in the history.
                      Loops are COUNTED in every arm (att.degenerate).
    timeout_as_cut    a request timeout before the deadline is treated like a cut turn
                      (retry, one rung lower) instead of failing the attempt."""
    t0 = time.time()
    # The early-submit wording and the nudge are ONE treatment: off means probe24's harness.
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": repl_prompt(task, early_submit=bool(nudge_every),
                                                        in_python=in_python)}]
    tools = make_tools(retain_functions)
    store = new_store() if retain_functions else None
    trim_state = {} if trim_drain else None
    rung, best_ok = -1, -1                   # rung -1 = the client's own reasoning effort

    def step_down():
        nonlocal rung
        if effort_ladder and rung < len(effort_ladder) - 1:
            rung += 1
            att.effort_steps += 1

    while att.turns < max_turns:
        with shared.lock:
            if shared.verified.get(att.task_id, 0) >= stop_after:
                att.status, att.note = "stopped", "task verified by a sibling"
                break
        if time.time() > deadline:
            att.status, att.note = "stopped", "deadline"
            break
        if att.gen_tokens >= max_gen_tokens:
            att.status, att.note = "failed", "token budget"
            break
        att.turns += 1
        # A request may not outlive the deadline by more than a minute.
        req_timeout = max(60.0, min(3600.0, deadline - time.time() + 60.0))
        # mini2arm: the text nudge was read and ignored. The forcesubmit treatment
        # makes the NEXT call a submit after this many python calls without one.
        force = (force_submit_after
                 and att.python_calls - att.python_at_submit >= force_submit_after)
        if force:
            att.forced += 1
        extra = {}
        if rung >= 0:            # only then, so clients without the kwarg keep working
            extra["template_kwargs"] = {"reasoning_effort": effort_ladder[rung]}
        r = client.chat(trim(messages, context_budget, keep_recent, trim_drain, trim_state),
                        tools=tools, seed=seed + att.turns,
                        timeout=req_timeout, max_tokens=turn_max_tokens,
                        tool_choice="submit" if force else None, **extra)
        if trim_state is not None:
            att.evictions = trim_state.get("evictions", 0)
        if "error" in r:
            if r.get("context_overflow") and keep_reasoning:
                # Kept reasoning can make even the protected recent turns too long for
                # the model; shed it (oldest first, newest last) before shrinking.
                held = [m for m in messages if reasoning_key in m]
                if held:
                    for m in held[:max(1, len(held) - 1)]:
                        m.pop(reasoning_key)
                    continue
            if r.get("context_overflow") and context_budget > 8000:
                context_budget = int(context_budget * 0.7)
                continue
            if time.time() >= deadline:
                # Requests are timed to end a minute after the deadline: this is the run
                # ending, not the attempt failing (ab3q36 logged 21 of these as failures).
                att.status, att.note = "stopped", "deadline (request cut)"
                break
            if timeout_as_cut and r.get("timeout"):
                att.timeouts += 1
                step_down()
                if messages[-1]["role"] != "user":
                    messages.append({"role": "user", "content": TIMEOUT_HINT})
                continue
            att.status, att.note = "failed", r["error"][:200]
            break
        att.gen_tokens += r["completion_tokens"]
        att.prompt_tokens += r["prompt_tokens"]
        cut = r["finish_reason"] == "length"
        loops = [k for k in ("reasoning", "content") if degenerate(r[k])]
        att.degenerate += bool(loops)
        att.transcript.append({"turn": att.turns, "t": round(time.time() - t0, 1),
                               "reasoning": r["reasoning"], "content": r["content"],
                               "calls": [c["name"] for c in r["tool_calls"]],
                               "finish": r["finish_reason"], "tokens": r["completion_tokens"],
                               **({"effort": effort_ladder[rung]} if rung >= 0 else {}),
                               **({"loop": loops} if loops else {})})
        if cut:
            step_down()
        content = r["content"] or ""
        if degenerate_guard and "content" in loops:
            content = DEGENERATE_STUB
        msg = {"role": "assistant", "content": content}
        if keep_reasoning and r["reasoning"] and not cut and "reasoning" not in loops:
            msg[reasoning_key] = r["reasoning"]
        calls = r["tool_calls"]
        if not calls:
            code = extract_code(r["content"])
            if code:                       # answered with a code block instead of submit
                calls = [{"id": f"implicit-{att.turns}", "name": "submit",
                          "arguments": {"code": code}, "implicit": True}]
            else:
                messages.append(msg)
                if r["finish_reason"] == "length":
                    # The thinking-cap treatment answers a cut with an order to act; the
                    # baseline keeps probe24's wording.
                    hint = THINKCAP_HINT if turn_max_tokens else "Your reply was cut off. Be more concise."
                    att.cuts += 1
                else:
                    hint = "Use the `python` tool to explore, or `submit` your transform program."
                messages.append({"role": "user", "content": hint})
                continue
        if not any(c.get("implicit") for c in calls):
            msg["tool_calls"] = [{"id": c["id"], "type": "function",
                                  "function": {"name": c["name"],
                                               "arguments": json.dumps(c["arguments"])}}
                                 for c in calls]
        messages.append(msg)
        done = False
        for c in calls:
            code = (c["arguments"] or {}).get("code") or ""
            if c["name"] == "python":
                att.python_calls += 1
                out = run_snippet(code, task, timeout=snippet_timeout, store=store)
                since = att.python_calls - att.python_at_submit
                if nudge_every and since and since % nudge_every == 0:
                    out += NUDGE.format(n=since)
                    att.nudges += 1
            elif c["name"] == "submit":
                att.submits += 1
                att.python_at_submit = att.python_calls
                full, added = with_kept(code, store) if store else (code, [])
                n_ok, test_out, out = verify(full, task)
                if added:
                    att.kept_used += 1
                    out += f"\n(Ran with your kept functions: {', '.join(added)}.)"
                ntr = len(task["train"])
                att.programs.append({"turn": att.turns, "code": full, "n_ok": n_ok,
                                     "n_train": ntr, "test": test_out})
                if n_ok > best_ok:
                    if best_ok >= 0 or n_ok > 0:
                        rung = -1            # progress: back to the client's own effort
                    best_ok = n_ok
                if n_ok == ntr and all(o is not None for o in test_out):
                    done = True
            else:
                out = f"Unknown tool {c['name']!r}. Tools: python, submit."
            att.transcript.append({"turn": att.turns, "tool": c["name"], "code": code,
                                   "output": out})
            if c.get("implicit"):
                messages.append({"role": "user", "content": out})
            else:
                messages.append({"role": "tool", "tool_call_id": c["id"], "content": out})
        if done:
            att.status = "verified"
            with shared.lock:
                shared.verified[att.task_id] = shared.verified.get(att.task_id, 0) + 1
            break
    else:
        att.status, att.note = "failed", "turn budget"
    att.wall = time.time() - t0
    return att


# run_attempt options a probe or submission config may set for the whole run
# (probe arms can set any run_attempt keyword).
AGENT_OPTS = ("turn_max_tokens", "in_python", "force_submit_after", "effort_ladder",
              "retain_functions", "keep_reasoning", "reasoning_key", "keep_recent", "trim_drain",
              "degenerate_guard", "timeout_as_cut", "snippet_timeout")


def attempt_record(a, task):
    return {"task_id": a.task_id, "idx": a.idx, "status": a.status,
            "note": a.note, "turns": a.turns, "python_calls": a.python_calls,
            "submits": a.submits, "nudges": a.nudges, "cuts": a.cuts, "forced": a.forced, "gen_tokens": a.gen_tokens,
            "prompt_tokens": a.prompt_tokens, "wall": a.wall,
            "effort_steps": a.effort_steps, "timeouts": a.timeouts, "degenerate": a.degenerate,
            "kept_used": a.kept_used, "evictions": a.evictions,
            "best_ok": max((p["n_ok"] for p in a.programs), default=-1),
            "n_train": len(task["train"]),
            "programs": a.programs, "transcript": a.transcript}


def solve_repl(tasks, client, attempts=2, concurrency=32, deadline=None, stop_after=1,
               log=print, on_done=None, task_kw=None, **kw):
    """Same return shape as agent.solve: (answers, report).

    `on_done(record)` is called as each attempt finishes, so a caller can persist
    results incrementally -- probe24 lost everything to one late exception."""
    deadline = deadline or float("inf")
    shared = _Shared()
    # attempt-major order: every task gets its first attempt before any gets a second
    atts = [Attempt(tid, k) for k in range(attempts) for tid in tasks]
    t0 = time.time()
    done_n = [0]

    def work(att):
        try:
            # task_kw lets arms of an A/B share one run: key -> per-arm overrides. An arm
            # may bring its own client (different sampling / chat-template kwargs).
            over = dict(kw, **(task_kw or {}).get(att.task_id, {}))
            arm_client = over.pop("client", client)
            run_attempt(att, tasks[att.task_id], arm_client, shared, deadline,
                        stop_after=stop_after, seed=1000 * att.idx, **over)
        except Exception as e:                  # one attempt must never sink the run
            import traceback
            att.status, att.note = "crashed", f"{type(e).__name__}: {e}"[:300]
            att.transcript.append({"crash": traceback.format_exc()[-2000:]})
        if on_done:
            try:
                on_done(attempt_record(att, tasks[att.task_id]))
            except Exception as e:
                log(f"on_done failed: {e}")
        with shared.lock:
            done_n[0] += 1
            n_ver = sum(1 for v in shared.verified.values() if v)
        log(f"[{time.time() - t0:6.0f}s] {att.task_id}#{att.idx} {att.status:8s} "
            f"turns {att.turns:2d} py {att.python_calls:2d} sub {att.submits} "
            f"tok {att.gen_tokens:6d} | done {done_n[0]}/{len(atts)} tasks verified {n_ver}")
        return att

    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        list(ex.map(work, atts))

    answers = {tid: aggregate(tid, tasks[tid], atts) for tid in tasks}
    report = {"wall_s": time.time() - t0,
              "attempts": [attempt_record(a, tasks[a.task_id]) for a in atts]}
    return answers, report


Writing /kaggle/working/agent/repl_agent.py


In [9]:
%%writefile /kaggle/working/agent/run_direct.py
"""Direct-answer mode: the model reasons (e.g. Qwen3.8 at reasoning_effort xhigh) and
writes the test output grid(s) itself -- no tools. Two samples per task give the two
attempts. This is how headline ARC-AGI-2 numbers for chat models are usually measured,
so it is the comparison the REPL agent has to beat.

    python run_direct.py '<json config>'

Config: base_url, model, task_ids, samples (2), concurrency, max_tokens, deadline_epoch,
temperature/top_p/top_k, template_kwargs, out, data_dir.
Writes <out> (JSON) and <out>.jsonl (one line per finished sample).
"""
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                       # noqa: E402
from grids import show                              # noqa: E402

# grids.SYSTEM tells the model it solves puzzles "by writing Python programs" -- right for
# the REPL agent, wrong when the answer is the grid itself.
DIRECT_SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles. Each puzzle has a few input->output "
    "example pairs that share one hidden transformation rule. Grids are 2D arrays of digits "
    "0-9 (colours); 0 is usually the background."
)

# A reply cut off before any grid (the thinking budget missing or ignored, or the answer
# itself cut) gets ONE short follow-up without thinking: the tail of its own reasoning,
# and an order to commit. An unanswered task scores nothing; a committed guess may score.
RESCUE = """{prompt}

## Your notes so far (they were cut off)
{notes}

Stop analysing now. Apply the rule you were converging on and give the final test output grid(s): each as rows of digits in its own ```grid code block, in test order, with nothing else in the block."""

DIRECT_INSTRUCTIONS = """Solve this ARC puzzle. The examples share one hidden transformation rule. Work out the rule, check it against every example, then apply it to the test input(s).

At the very end of your answer, give each test output grid as rows of digits inside its own ```grid code block, in test order, with nothing else in the block."""


def direct_prompt(task):
    parts = [DIRECT_INSTRUCTIONS, ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    return "\n".join(parts).rstrip()


def parse_grids(text, n_tests):
    """The last n_tests digit-only code blocks (```grid preferred, any fence accepted)."""
    blocks = re.findall(r"```(?:grid)?[^\n]*\n(.*?)```", text or "", re.S)
    grids = []
    for b in blocks:
        rows = [r.strip() for r in b.strip().splitlines() if r.strip()]
        rows = [r.replace(" ", "").replace(",", "") for r in rows]
        if rows and all(r.isdigit() for r in rows) and len({len(r) for r in rows}) == 1:
            grids.append([[int(c) for c in r] for r in rows])
    return grids[-n_tests:] if len(grids) >= n_tests else None


def answer(client, task, seed, deadline, max_tokens=None, rescue_tokens=8192, notes_chars=16000):
    """One direct answer to `task`: reason, then write the test grid(s). Returns a record;
    rec["grids"] is the list of test grids, or None when nothing parseable came back."""
    n = len(task["test"])
    prompt = direct_prompt(task)

    def timeout():
        return max(60.0, min(4 * 3600, deadline - time.time() + 60))

    mt = max_tokens or client.max_tokens
    for _ in range(3):
        r = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                         {"role": "user", "content": prompt}],
                        seed=seed, max_tokens=mt, timeout=timeout())
        if not (r.get("context_overflow") and mt > 8192):
            break
        mt = int(mt * 0.75)                    # a long prompt: shrink the reply, never give up
    if "error" in r:
        return {"status": "error: " + r["error"][:200], "grids": None, "gen_tokens": 0}
    rec = {"finish": r["finish_reason"], "gen_tokens": r["completion_tokens"],
           "reasoning_chars": len(r["reasoning"] or ""),
           "content_tail": (r["content"] or "")[-1500:],
           "reasoning_tail": (r["reasoning"] or "")[-3000:]}
    grids = parse_grids(r["content"], n)
    if grids:
        return dict(rec, status="answered", grids=grids)
    notes = ((r["reasoning"] or "") + "\n" + (r["content"] or "")).strip()
    if notes and deadline - time.time() > 120:
        r2 = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                          {"role": "user", "content": RESCUE.format(prompt=prompt,
                                                                    notes=notes[-notes_chars:])}],
                         seed=seed + 1, max_tokens=rescue_tokens, timeout=timeout(),
                         template_kwargs={"enable_thinking": False, "reasoning_effort": "low"})
        if "error" in r2:
            rec["rescue_error"] = r2["error"][:200]
        else:
            rec["gen_tokens"] += r2["completion_tokens"]
            rec["rescue_finish"] = r2["finish_reason"]
            rec["rescue_tail"] = ((r2["content"] or "") or (r2["reasoning"] or ""))[-1500:]
            # Without thinking, a reasoning parser may still file the reply as reasoning.
            grids = parse_grids(r2["content"], n) or parse_grids(r2["reasoning"], n)
            if grids:
                return dict(rec, status="rescued", grids=grids)
    return dict(rec, status="no grid", grids=None)


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    import glob
    ch = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_challenges.json", recursive=True)[0]))
    sol = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_solutions.json", recursive=True)[0]))
    ids = cfg["task_ids"]
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 49152),
                        temperature=cfg.get("temperature", 1.0), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"), timeout=4 * 3600)
    out = cfg.get("out", "/kaggle/working/direct_results.json")
    jsonl = out.replace(".json", ".jsonl")
    deadline = cfg.get("deadline_epoch", float("inf"))
    lock = threading.Lock()
    recs = []
    jobs = [(t, k) for k in range(cfg.get("samples", 2)) for t in ids]   # sample-major

    def work(job):
        t, k = job
        rec = {"task_id": t, "sample": k}
        if time.time() > deadline:
            rec["status"] = "skipped: deadline"
        else:
            ts = time.time()
            rec.update(answer(client, ch[t], 1000 * k + 7, deadline,
                              rescue_tokens=cfg.get("rescue_tokens", 8192)))
            rec["wall"] = round(time.time() - ts, 1)
            grids = rec["grids"]
            rec["correct"] = [g == s for g, s in zip(grids, sol[t])] if grids else None
        with lock:
            recs.append(rec)
            with open(jsonl, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"[{time.time() - t0:6.0f}s] {t}#{k} {rec['status']:10s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} "
                  f"correct {rec.get('correct')}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 16)) as ex:
        list(ex.map(work, jobs))

    # pass@2 per task: any test output matched by either sample (fractional per test).
    per = {}
    for t in ids:
        n = len(sol[t])
        hit = [False] * n
        for r in recs:
            if r["task_id"] == t and r.get("correct"):
                hit = [h or c for h, c in zip(hit, r["correct"])]
        per[t] = sum(hit) / n
    tok = [r["gen_tokens"] for r in recs if "gen_tokens" in r]
    summary = {"config": cfg, "per_task": per, "score": sum(per.values()), "n_tasks": len(ids),
               "samples": len(recs), "mean_gen_tokens": sum(tok) / max(len(tok), 1),
               "cut_at_max_tokens": sum(1 for r in recs if r.get("finish") == "length"),
               "wall_s": time.time() - t0, "records": recs}
    json.dump(summary, open(out, "w"))
    print(f"\nDIRECT SCORE {summary['score']:.2f} / {len(ids)}  mean tokens {summary['mean_gen_tokens']:.0f}  "
          f"cut {summary['cut_at_max_tokens']}  wall {summary['wall_s']:.0f}s", flush=True)
    print("per task:", {t: round(v, 2) for t, v in per.items()}, flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


Writing /kaggle/working/agent/run_direct.py


In [10]:
%%writefile /kaggle/working/agent/run_submit.py
"""Competition run: the REPL agent on the hidden test set, writing submission.json.

    python run_submit.py '<json config>'

Safety first, because a crashed rerun wastes a day's submission:
  * the notebook writes a valid FALLBACK submission (each test input copied into
    both attempts) before anything heavy runs, via write_fallback();
  * as every attempt finishes, the submission is rewritten atomically with the
    best answers so far;
  * the agent stops at a hard deadline well inside the 12 h limit.

Budget: the per-attempt token cap is the estimated token supply (throughput x
time) divided by the number of hidden tasks, clamped -- the task count is only
known at run time (the public placeholder has 240).
"""
import glob
import json
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                      # noqa: E402
from repl_agent import solve_repl, AGENT_OPTS      # noqa: E402

SUB = "/kaggle/working/submission.json"


def find(name, root="/kaggle/input"):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def write_atomic(obj, path=SUB):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, path)


def write_fallback(challenges, path=SUB):
    """A valid submission with no model at all: every attempt is the test input."""
    write_atomic({t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in task["test"]]
                  for t, task in challenges.items()}, path)


def ranked_answers(records, task):
    """Per test input: candidate grids best first. Verified programs vote; then the
    unverified programs by train examples reproduced. Mirrors agent.aggregate."""
    out = []
    ntr = len(task["train"])
    for j in range(len(task["test"])):
        score = {}
        grids = {}
        for r in records:
            for p in r["programs"]:
                g = p["test"][j] if j < len(p["test"]) else None
                if g is None:
                    continue
                k = json.dumps(g)
                grids[k] = g
                verified = p["n_ok"] == ntr
                s = score.get(k, (0, 0, 0))
                score[k] = (s[0] + verified, max(s[1], p["n_ok"]), s[2] + 1)
        out.append([grids[k] for k in sorted(score, key=lambda k: score[k], reverse=True)])
    return out


def direct(cfg, tasks, client, deadline, sub, sub_path, t0):
    """Direct answers (run_direct.answer): every task's sample 0 before any sample 1, each
    task in `tasks` order. attempt_1 = the first answer; attempt_2 = the next DIFFERENT one."""
    from run_direct import answer
    lock = threading.Lock()
    got = {}
    jobs = [(t, k) for k in range(cfg.get("samples", 2)) for t in tasks]
    log = open(cfg.get("log_path", "/kaggle/working/direct_attempts.jsonl"), "a")
    count = {}

    def work(job):
        t, k = job
        if deadline - time.time() < cfg.get("min_start_s", 600):
            rec = {"status": "skipped: deadline", "grids": None, "gen_tokens": 0}
        else:
            ts = time.time()
            rec = answer(client, tasks[t], 1000 * k + 7, deadline, max_tokens=cfg.get("max_tokens"),
                         rescue_tokens=cfg.get("rescue_tokens", 8192))
            rec["wall"] = round(time.time() - ts, 1)
        with lock:
            if rec["grids"]:
                got.setdefault(t, []).append(rec["grids"])
                for j, x in enumerate(tasks[t]["test"]):
                    cands = []
                    for grids in got[t]:
                        if grids[j] not in cands:
                            cands.append(grids[j])
                    sub[t][j] = {"attempt_1": cands[0],
                                 "attempt_2": cands[1] if len(cands) > 1 else x["input"]}
                write_atomic(sub, sub_path)
            st = rec["status"].split(":")[0]
            count[st] = count.get(st, 0) + 1
            log.write(json.dumps(dict({k2: v for k2, v in rec.items() if k2 != "grids"},
                                      task_id=t, sample=k)) + "\n")
            log.flush()
            print(f"[{time.time() - t0:6.0f}s] {t}#{k} {rec['status'][:40]:12s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} | {count}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 10)) as ex:
        list(ex.map(work, jobs))
    log.close()
    print(f"done: {len(got)}/{len(tasks)} tasks answered, {count}, {time.time() - t0:.0f}s", flush=True)


def main(cfg):
    t0 = time.time()
    rerun = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
    sub_path = cfg.get("sub_path", SUB)
    ch = json.load(open(find(cfg.get("challenges", "arc-agi_test_challenges.json"),
                             cfg.get("data_dir", "/kaggle/input"))))
    if not rerun and not cfg.get("full_interactive"):
        # Interactive run (should not happen -- versions are Quick Saved): stay tiny.
        ch = {t: ch[t] for t in sorted(ch)[:cfg.get("interactive_tasks", 3)]}
    if not os.path.exists(sub_path):
        write_fallback(ch, sub_path)
    n = len(ch)
    deadline = cfg["deadline_epoch"]
    avail = max(0.0, deadline - time.time())
    supply = cfg.get("tok_per_s", 140) * avail
    cap = int(min(cfg.get("max_cap", 60000), max(cfg.get("min_cap", 15000), supply / max(n, 1) * 1.3)))
    print(f"rerun={rerun} tasks={n} available {avail / 3600:.2f} h, est supply {supply / 1e6:.1f}M "
          f"tokens -> per-attempt cap {cap}", flush=True)

    # Cheapest prompts first: if the deadline bites, the long tasks are the ones lost.
    order = sorted(ch, key=lambda t: len(json.dumps(ch[t])))
    tasks = {t: ch[t] for t in order}
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 16384),
                        temperature=cfg.get("temperature", 0.6), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        reasoning_effort=cfg.get("reasoning_effort"),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"))

    sub = json.load(open(sub_path))
    if cfg.get("mode") == "direct":
        return direct(cfg, tasks, client, deadline, sub, sub_path, t0)

    lock = threading.Lock()
    done = {}

    def on_done(rec):
        with lock:
            done.setdefault(rec["task_id"], []).append(rec)
            task = tasks[rec["task_id"]]
            for j, cands in enumerate(ranked_answers(done[rec["task_id"]], task)):
                inp = task["test"][j]["input"]
                a1 = cands[0] if cands else inp
                a2 = cands[1] if len(cands) > 1 else inp
                sub[rec["task_id"]][j] = {"attempt_1": a1, "attempt_2": a2}
            write_atomic(sub, sub_path)

    _, report = solve_repl(
        tasks, client, attempts=cfg.get("attempts", 1), concurrency=cfg.get("concurrency", 24),
        deadline=deadline, stop_after=1, max_turns=cfg.get("max_turns", 40), max_gen_tokens=cap,
        context_budget=cfg.get("context_budget", 24000), nudge_every=cfg.get("nudge_every", 0),
        turn_max_tokens=cfg.get("turn_max_tokens"), in_python=cfg.get("in_python", False),
        log=lambda m: print(m, flush=True), on_done=on_done,
        **{k: cfg[k] for k in AGENT_OPTS if k in cfg and k not in ("turn_max_tokens", "in_python")})

    ver = sum(1 for a in report["attempts"] if a["status"] == "verified")
    print(f"done: {ver}/{n} tasks verified, {sum(a['gen_tokens'] for a in report['attempts'])} "
          f"tokens, {time.time() - t0:.0f}s", flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


Writing /kaggle/working/agent/run_submit.py


In [11]:
%%writefile /kaggle/working/agent/submit_main.py
"""Everything the competition rerun does after the fallback submission is written:
install the vLLM wheelhouse, start the server, run the agent. It NEVER raises --
an exception in a rerun fails the submission even when submission.json exists --
so every stage is caught and logged, and the fallback stands if a stage fails.

    python submit_main.py '<json config>'
"""
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = Path("/kaggle/working")
SITE = WORK / "vllm-site-packages"
LOG = WORK / "vllm-server.log"


def one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])


def install(cfg):
    wh = one("/kaggle/input/**/arc3-vllm-h100-wheelhouse-v3/**/requirements.lock").parent
    t = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wh),
                    "--requirement", str(wh / "requirements.lock"), "--target", str(SITE),
                    "--upgrade", "--ignore-installed", "--only-binary", ":all:", "--no-compile",
                    "--disable-pip-version-check", "--no-warn-conflicts", "-q"], check=True)
    print(f"wheelhouse installed in {time.time() - t:.0f}s", flush=True)
    if cfg.get("overlay"):
        wheels = sorted(glob.glob(cfg["overlay"], recursive=True))
        if not wheels:
            raise FileNotFoundError(cfg["overlay"])
        for pat in ("transformers", "transformers-*.dist-info", "huggingface_hub",
                    "huggingface_hub-*.dist-info"):
            for p in SITE.glob(pat):
                shutil.rmtree(p) if p.is_dir() else p.unlink()
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--target", str(SITE), "--upgrade", "--force-reinstall", "--no-compile",
                        "--disable-pip-version-check", "-q", *wheels], check=True)
        print("overlay installed:", [Path(w).name for w in wheels], flush=True)


def start_server(cfg, extra):
    model = one(cfg["model_glob"]).parent
    env = dict(os.environ, PYTHONPATH=str(SITE), USE_TF="0", TRANSFORMERS_NO_TF="1",
               TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1",
               LIBRARY_PATH="/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", ""))
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
           "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", "1234",
           "--tensor-parallel-size", "4", "--enable-auto-tool-choice", *cfg["parsers"],
           "--generation-config", "vllm", "--enable-prefix-caching",
           "--max-model-len", str(cfg.get("max_model_len", 65536)), "--gpu-memory-utilization", "0.90",
           "--max-num-seqs", str(cfg.get("max_num_seqs", 32)), *extra]
    print(" ".join(cmd), flush=True)
    t = time.time()
    # "w": a retry must not see the previous attempt's "EngineCore failed".
    proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    while time.time() - t < 1500:
        text = LOG.read_text(errors="replace") if LOG.exists() else ""
        if proc.poll() is not None or "EngineCore failed" in text:
            break
        try:
            urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=5)
            print(f"server ready after {time.time() - t:.0f}s", flush=True)
            return proc
        except Exception:
            time.sleep(10)
    print("server FAILED to start; tail:\n" + "\n".join(
        LOG.read_text(errors="replace").splitlines()[-30:]), flush=True)
    kill(proc)
    return None


def kill(proc):
    """Kill the server's whole process group -- engine workers hold GPU memory -- and wait
    for the GPUs to drain, so a restart does not hit out-of-memory."""
    import signal
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        proc.kill()
    try:
        proc.wait(timeout=60)
    except Exception:
        pass
    for _ in range(36):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.split()
        if used and all(int(u) < 1500 for u in used if u.strip().isdigit()):
            return
        time.sleep(5)


def server_ladder(cfg):
    """Option sets to try in order. An explicit cfg["server_ladder"] wins (plain serving is
    appended if missing); otherwise server_extra + server_try -> server_extra -> plain."""
    if cfg.get("server_ladder"):
        ladder = [list(x) for x in cfg["server_ladder"]]
        return ladder if [] in ladder else ladder + [[]]
    base = cfg.get("server_extra", [])
    ladder = [base + cfg["server_try"]] if cfg.get("server_try") else []
    return ladder + [base] + ([[]] if base else [])


def main(cfg):
    proc = None
    try:
        install(cfg)
        # A ladder of server option sets, most ambitious first; the last is plain serving.
        # Untested-on-L4 options (MTP, --reasoning-config) must never cost the whole run.
        ladder = server_ladder(cfg)
        for i, extra in enumerate(ladder):
            proc = start_server(cfg, extra)
            if proc is not None:
                print(f"server up with option set {i}: {extra}", flush=True)
                break
            print(f"option set {i} failed -> next", flush=True)
            time.sleep(20)
        if proc is None:
            return
        sys.path.insert(0, HERE)
        import run_submit
        run_submit.main(dict(cfg, base_url="http://127.0.0.1:1234/v1", model="qwen"))
    except BaseException:
        print("submit_main caught:\n" + traceback.format_exc(), flush=True)
    finally:
        if proc is not None:
            kill(proc)
        shutil.rmtree(SITE, ignore_errors=True)


if __name__ == "__main__":
    try:
        main(json.loads(sys.argv[1]))
    except BaseException:
        print(traceback.format_exc(), flush=True)
    sys.exit(0)


Writing /kaggle/working/agent/submit_main.py


In [12]:
import os
os.makedirs("/kaggle/working/floor", exist_ok=True)
print("floor dir ready")

floor dir ready


In [13]:
%%writefile /kaggle/working/floor/arc_envfix.py

import os, sys
def apply():
    util = [p for p in sys.path if p.startswith("/kaggle/usr/lib/")]
    if sys.version_info[:2] == (3, 11) or not util:
        return []
    for p in util:
        while p in sys.path:
            sys.path.remove(p)
    os.environ["PYTHONPATH"] = os.pathsep.join(
        x for x in os.environ.get("PYTHONPATH", "").split(os.pathsep) if x and not x.startswith("/kaggle/usr/lib/"))
    for m in [m for m in sys.modules if m.split(".")[0] in ("numpy", "torch", "unsloth")]:
        del sys.modules[m]
    return util
STRIPPED = apply()


Writing /kaggle/working/floor/arc_envfix.py


In [14]:
%%writefile /kaggle/working/floor/arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing /kaggle/working/floor/arc_loader.py


In [15]:
%%writefile /kaggle/working/floor/arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:  # a worker killed mid-write leaves a truncated file
                print(f"*** skipping unreadable result file {key}: {e!r}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

Writing /kaggle/working/floor/arc_decoder.py


In [16]:
%%writefile /kaggle/working/floor/arc_sched.py
"""Cost model and budget pacing for the 4-worker TTT pipeline.

The cost model was fitted on 24 ARC-AGI-2 evaluation tasks run with the baseline
configuration (128 TTT sequences, 16 views, DFS p>=0.2) on one L4:
  train  ~ 19.2 + 0.0809*S + 4.16e-6*S^2   seconds per 128 steps (S = training sequence tokens)
  decode ~ 185 + 0.588*OT - 0.0381*OT*S/1000   seconds for 16 views (OT = predicted output tokens)
Training is predicted within ~10 s, decoding only within ~230 s, so the pacer calibrates
a global factor kappa = actual / predicted online and re-plans every task from the
remaining wall time.
"""
import time

import numpy as np

TRAIN_COEF = (19.2334, 0.0809330, 4.16404e-6)
DECODE_COEF = (185.248, 0.588264, -0.0381164)
SETUP_S = 15.0

# Level 0 = baseline config. The baseline stops starting decode batches 1200 s into a task
# and lets the running batch finish (up to its DFS window), so tasks can last ~1700 s; the
# cap reproduces that. Boost levels are disabled: the v8 run with them scored 26.39.
LEVELS = {
    0: dict(n_train_aug=16, extra_views=False, factor=1.00, cap=1700.0),
    1: dict(n_train_aug=16, extra_views=True,  factor=1.30, cap=1700.0),
    2: dict(n_train_aug=24, extra_views=True,  factor=1.55, cap=1900.0),
}
LEVEL1_RHO = float("inf")
LEVEL2_RHO = float("inf")
KAPPA_PRIOR_WEIGHT = 1500.0

PASS_SEEDS = {
    "A":  dict(train_seed=1, lora_seed=42,  eval_seed=2, suffix=""),
    "B":  dict(train_seed=17, lora_seed=137, eval_seed=29, suffix=".p2"),
}


def seq_tokens(task):
    return sum(len(g) * (len(g[0]) + 1) + 6 for p in task["train"] for g in (p["input"], p["output"]))


def predicted_output_tokens(task, test_item):
    ins = [np.shape(p["input"]) for p in task["train"]]
    outs = [np.shape(p["output"]) for p in task["train"]]
    x = np.shape(test_item["input"])
    if all(a == b for a, b in zip(ins, outs)):
        o = x
    elif len(set(outs)) == 1:
        o = outs[0]
    else:
        ry = np.mean([b[0] / a[0] for a, b in zip(ins, outs)])
        rx = np.mean([b[1] / a[1] for a, b in zip(ins, outs)])
        o = (max(1, min(30, round(x[0] * ry))), max(1, min(30, round(x[1] * rx))))
    return o[0] * (o[1] + 1) + 1


def est_train_seconds(task, n_train_aug=16):
    S = min(seq_tokens(task), 8192)
    per128 = TRAIN_COEF[0] + TRAIN_COEF[1] * S + TRAIN_COEF[2] * S * S
    return per128 * n_train_aug / 16


def est_decode_seconds(task):
    S = min(seq_tokens(task), 8192)
    total = 0.0
    for t in task["test"]:
        ot = predicted_output_tokens(task, t)
        total += DECODE_COEF[0] + DECODE_COEF[1] * ot + DECODE_COEF[2] * ot * S / 1000
    # the fit is per task (one test output for most tasks); extra outputs add less than a full task
    n = len(task["test"])
    total = total * (1 + 0.6 * (n - 1)) / n
    return float(min(max(total, 60.0), 1300.0))


def est_task_seconds(task):
    return SETUP_S + est_train_seconds(task) + est_decode_seconds(task)


def output_uncertainty(u):
    """0 = settled, higher = worth another independent TTT run."""
    if u["c1"] == 0:
        return 0.25  # no candidate at all: a second run rarely recovers these
    if u["c1"] >= max(6, u["views"] // 2) and u["c1"] >= 2 * u["c2"] + 2:
        return 0.0
    return 1.0 - u["c1"] / max(u["views"], 1) + 0.25 * min(u["n_unique"], 4) / 4


class Pacer:
    """Hands out tasks and per-task wall budgets to one worker.

    shared: dict of Manager proxies created by starter.py
      lock, order (list of keys, cheap first), est (key -> base seconds),
    tasks: key -> ARC task (train/test inputs), for the training-time floor
      state (qi, rem, act, exp), busy (rank -> busy-until ts),
      unc (key -> per-output uncertainty), tail_done (key -> 1), failed (key -> 1)
    """

    def __init__(self, shared, rank, n_workers, end_time, tasks):
        self.s = shared
        self.tasks = tasks
        self.rank = rank
        self.n_workers = n_workers
        self.end_time = end_time

    def kappas(self):
        st = self.s["state"]
        kt = (KAPPA_PRIOR_WEIGHT + st["tr_act"]) / (KAPPA_PRIOR_WEIGHT + st["tr_exp"])
        kd = (KAPPA_PRIOR_WEIGHT + st["dec_act"]) / (KAPPA_PRIOR_WEIGHT + st["dec_exp"])
        return kt, kd

    def kappa(self):
        """Overall actual/predicted ratio for a typical task mix."""
        kt, kd = self.kappas()
        st = self.s["state"]
        return (kt * st["rem_tr0"] + kd * st["rem_dec0"]) / max(st["rem_tr0"] + st["rem_dec0"], 1.0)

    def task_seconds(self, key, level=0):
        kt, kd = self.kappas()
        lv = LEVELS[level]
        t = self.tasks[key]
        dec_factor = 1.35 if lv["extra_views"] else 1.0
        return SETUP_S + kt * est_train_seconds(t, lv["n_train_aug"]) + kd * est_decode_seconds(t) * dec_factor

    def available(self, now):
        others = 0.0
        for r, until in self.s["busy"].items():
            if r != self.rank:
                others += max(0.0, until - now)
        return self.n_workers * max(0.0, self.end_time - now) - others

    def _plan(self, key, pass_name, rem_tr, rem_dec, now):
        kt, kd = self.kappas()
        A = self.available(now)
        task = self.tasks[key]
        base = self.task_seconds(key, 0)
        D = kt * rem_tr + kd * rem_dec + base
        rho = A / max(D, 1.0)
        if pass_name == "A":
            level = 2 if rho >= LEVEL2_RHO else (1 if rho >= LEVEL1_RHO else 0)
        else:
            level = 0
        lv = LEVELS[level]
        expected = self.task_seconds(key, level)
        train_floor = SETUP_S + kt * est_train_seconds(task, lv["n_train_aug"]) + 90.0
        if rho >= 1.0:
            # on schedule: give the task the full baseline allowance, never less
            budget = lv["cap"]
        else:
            # behind schedule: shrink this task's share in proportion
            budget = min(lv["cap"], 1.2 * base * max(rho, 0.25))
        budget = max(budget, train_floor)
        budget = min(budget, self.end_time - now - 20.0)
        plan = dict(task=key, pass_name=pass_name, level=level, est=self.s["est"][key], kappa=kd, rho=rho,
                    budget=budget, expected=min(budget, expected),
                    est_train=est_train_seconds(task, lv["n_train_aug"]), est_dec=est_decode_seconds(task) * (1.35 if lv["extra_views"] else 1.0),
                    n_train_aug=lv["n_train_aug"], extra_views=lv["extra_views"],
                    **PASS_SEEDS[pass_name])
        return plan, train_floor

    def next_task(self):
        while True:
            now = time.time()
            if now > self.end_time - 120:
                return None
            with self.s["lock"]:
                st = self.s["state"]
                qi = st["qi"]
                order = self.s["order"]
                if qi < len(order):
                    key = order[qi]
                    task = self.tasks[key]
                    st["qi"] = qi + 1
                    st["rem_tr0"] = st["rem_tr0"] - est_train_seconds(task)
                    st["rem_dec0"] = st["rem_dec0"] - est_decode_seconds(task)
                    plan, train_floor = self._plan(key, "A", st["rem_tr0"], st["rem_dec0"], now)
                    if now + train_floor > self.end_time - 20:
                        print(f"[Rank {self.rank}] skip {key}: not enough time to train "
                              f"(needs {train_floor:.0f}s, {self.end_time - now:.0f}s left)")
                        continue
                    self.s["busy"][self.rank] = now + plan["expected"]
                    return plan
                # main queue exhausted -> tail pass on the most uncertain finished tasks
                plan = self._pick_tail(now)
                if plan is None:
                    self.s["busy"][self.rank] = now
                    return None
                self.s["busy"][self.rank] = now + plan["expected"]
                return plan

    def _pick_tail(self, now):
        left = self.end_time - now - 30.0
        best, best_score = None, 0.0
        done = self.s["tail_done"]
        failed = dict(self.s["failed"])
        for key, unc in list(self.s["unc"].items()) + [(k, None) for k in failed]:
            if key in done:
                continue
            est = self.task_seconds(key, 0)
            if est * 1.1 > left:
                continue
            if unc is None:
                score = 2.0
            else:
                us = [output_uncertainty(u) for u in unc.values()]
                score = float(np.mean(us)) if us else 0.0
            if score <= 0.05:
                continue
            score = score / (est / 600.0 + 0.5)
            if score > best_score:
                best, best_score = key, score
        if best is None:
            return None
        done[best] = 1
        est = self.task_seconds(best, 0)
        pass_name = "A" if best in failed else "B"
        plan, _ = self._plan(best, pass_name, 0.0, 0.0, now)
        plan["budget"] = min(max(plan["budget"], est), left)
        plan["expected"] = min(plan["budget"], est)
        return plan

    def done(self, plan, actual, info):
        with self.s["lock"]:
            st = self.s["state"]
            if info is not None and "train_s" in info:
                st["tr_act"] = st["tr_act"] + max(info["train_s"] - SETUP_S, 1.0)
                st["tr_exp"] = st["tr_exp"] + plan["est_train"]
                dec = max(actual - info["train_s"], 1.0)
                if info.get("timed_out", False):
                    dec *= 1.3  # censored: the decode would have run longer
                st["dec_act"] = st["dec_act"] + dec
                st["dec_exp"] = st["dec_exp"] + plan["est_dec"]
            self.s["busy"][self.rank] = time.time()
            if info is None:
                if plan["pass_name"] == "A":
                    self.s["failed"][plan["task"]] = 1
            elif plan["pass_name"] == "A":
                self.s["unc"][plan["task"]] = info["unc"]
                if plan["task"] in self.s["failed"]:
                    del self.s["failed"][plan["task"]]


Writing /kaggle/working/floor/arc_sched.py


In [17]:
%%writefile /kaggle/working/floor/arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import json
import time
import zlib
import traceback
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

from arc_sched import Pacer

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default


# Defaults reproduce the public NVARC baseline (128 TTT sequences, 16 views, DFS p>=0.2).
CFG = dict(
    model_path       = _env("ARC_MODEL_PATH", "", str),
    out_dir          = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    telemetry        = _env("ARC_TELEMETRY", "/kaggle/working/telemetry.jsonl", str),
    min_prob         = _env("ARC_MIN_PROB", 0.2, float),
    extra_min_prob   = _env("ARC_EXTRA_MIN_PROB", 0.12, float),
    window_per_view  = _env("ARC_WINDOW_PER_VIEW", 135.0, float),   # baseline: 540 s per 4-view batch
    max_rows         = _env("ARC_MAX_ROWS", 8, int),
    rows8_max_ctx    = _env("ARC_ROWS8_MAX_CTX", 3600, int),
    merge_lora       = _env("ARC_MERGE_LORA", 0, int),   # off: bf16 rounding of W + B@A erodes the TTT update
    bench            = _env("ARC_BENCH", 0, int),
)


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


DFS_STEPS = [0]


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, window) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < window and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        DFS_STEPS[0] += 1
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            window=window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        window=window,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def stable_seed(key, offset=0):
    # The baseline used hash(bk), which is salted per process (PYTHONHASHSEED unset);
    # crc32 keeps the 8 rescoring views identical across workers and reruns.
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


def resolve_model_dir():
    if CFG["model_path"] and os.path.isfile(os.path.join(CFG["model_path"], "config.json")):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    import glob
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if "grids15" in d and os.path.isfile(os.path.join(d, "tokenizer.json")):
            return d
    raise FileNotFoundError("qwen3_4b_grids15_sft139 model directory not found under /kaggle/input")


# ---------------------------------------------------------------------------
# Inference-time LoRA merge.
# Unsloth's fast inference path applies the r=256 adapter as two extra matmuls per
# projection (14 per layer). After TTT we fold B@A into the frozen base weights and mark
# the layer merged (unsloth's get_lora_parameters and PEFT both skip the adapter then).
# The original bf16 base weights are restored bit-exactly from a CPU snapshot before
# the next task trains, so no rounding drift accumulates across tasks.
# ---------------------------------------------------------------------------
class LoraMerger:

    def __init__(self, model, snapshot_path=None):
        from peft.tuners.lora import LoraLayer
        self.layers = []
        for name, module in model.named_modules():
            if isinstance(module, LoraLayer) and isinstance(getattr(module, "base_layer", None), torch.nn.Linear):
                if "default" in module.lora_A and isinstance(getattr(module, "merged_adapters", None), list):
                    self.layers.append((name, module))
        if snapshot_path is None:
            self.snapshot = {
                name: module.base_layer.weight.detach().to("cpu", copy=True)
                for name, module in self.layers
            }
        else:
            self.snapshot = self._shared_snapshot(snapshot_path)
            # the file was written by whichever worker got there first: check it is ours
            for name, module in self.layers[:: max(1, len(self.layers) // 8)]:
                if not torch.equal(module.base_layer.weight.detach().cpu(), self.snapshot[name]):
                    raise RuntimeError(f"base-weight snapshot mismatch at {name}")
        self.merged = False

    def _shared_snapshot(self, path):
        """One copy of the frozen base weights for all workers: written once to local disk,
        memory-mapped by every worker so the page cache is shared (4 private copies would
        cost ~29 GB of host RAM)."""
        lock = path + ".lock"
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
            owner = True
        except FileExistsError:
            owner = False
        if owner:
            state = {name: module.base_layer.weight.detach().to("cpu", copy=True) for name, module in self.layers}
            torch.save(state, path + ".tmp")
            del state
            os.replace(path + ".tmp", path)
        else:
            t0 = time.time()
            while not os.path.exists(path):
                if time.time() - t0 > 300:
                    raise TimeoutError("shared base-weight snapshot was never written")
                time.sleep(2)
        return torch.load(path, mmap=True, weights_only=True)

    @torch.no_grad()
    def merge(self):
        self.merged = True  # set first: restore() must run even if merging stops half-way
        for name, module in self.layers:
            W = module.base_layer.weight
            A = module.lora_A["default"].weight.float()
            B = module.lora_B["default"].weight.float()
            s = module.scaling["default"]
            W.copy_((W.float() + (B @ A) * s).to(W.dtype))
            module.merged_adapters.append("default")

    @torch.no_grad()
    def restore(self):
        if not self.merged:
            return
        for name, module in self.layers:
            module.base_layer.weight.copy_(self.snapshot[name], non_blocking=False)
            module.merged_adapters.clear()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.merged = False


def view_geometry(subkey):
    ops = subkey.split(".")[1:]
    transposed = len(ops) > 0 and ops[0] == "transpose"
    rot = sum(1 for op in ops if op == "rot90")
    return transposed, rot


def make_batches(eval_ds, formatter, tokenizer, max_rows, rows8_max_ctx):
    """Group decoded views into DFS batches.

    Turbo DFS shares one position id across rows, so every row of a batch must have the
    same prompt length; views are grouped by their exact token count (shape class
    {id, rot180, T.rot90, T.rot270} vs {rot90, rot270, T, T.rot180}, minus any
    examples cut_to_len dropped). Short prompts decode 8 rows per batch, long ones 4.
    Batches are interleaved across test outputs so a time cap leaves every output
    with views from both shape classes, as in the baseline order."""
    per_test = defaultdict(lambda: defaultdict(list))
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        tokens = tokenizer.encode(eval_ds.get(subkey, formatter)["input"])
        per_test[test_id][len(tokens)].append((subkey, tokens))
    chunks_per_test = {}
    for test_id, groups in per_test.items():
        chunks = []
        # largest group first == the class containing the identity view in normal cases
        for length, items in sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):
            rows = max_rows if length <= rows8_max_ctx else min(4, max_rows)
            for i in range(0, len(items), rows):
                chunks.append(items[i:i + rows])
        # alternate shape classes: A1, B1, A2, B2 ...
        half = defaultdict(list)
        for c in chunks:
            half[len(c[0][1])].append(c)
        ordered = []
        lists = list(half.values())
        for i in range(max(len(l) for l in lists)):
            for l in lists:
                if i < len(l):
                    ordered.append(l[i])
        chunks_per_test[test_id] = ordered
    batches = []
    n = max(len(c) for c in chunks_per_test.values())
    for i in range(n):
        for test_id in sorted(chunks_per_test):
            if i < len(chunks_per_test[test_id]):
                batches.append(chunks_per_test[test_id][i])
    return batches


def output_confident(results_by_bk, bk, n_views):
    """True if one candidate clearly dominates the views decoded so far."""
    counts = defaultdict(int)
    for sol in results_by_bk.get(bk, []):
        counts[sol] += 1
    if not counts:
        return False
    top = sorted(counts.values(), reverse=True)
    c1 = top[0]
    c2 = top[1] if len(top) > 1 else 0
    return c1 >= max(6, n_views // 2) and c1 >= 2 * c2 + 2


def tel(path, rec):
    try:
        with open(path, "a") as f:
            f.write(json.dumps(rec) + "\n")
    except Exception:
        pass


def worker(rank, end_time, shared, n_workers, test_path):

    # turbo_dfs recurses once per generated token (up to 931 for a 30x30 grid)
    import sys
    sys.setrecursionlimit(max(sys.getrecursionlimit(), 6000))

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    bench_pending = bool(CFG["bench"])

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=resolve_model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    merger = None
    if CFG["merge_lora"]:
        try:
            merger = LoraMerger(model, snapshot_path=os.getenv("ARC_SNAPSHOT", "/tmp/arc_base_weights.pt"))
            print(f"[Rank {rank}] LoRA merge enabled over {len(merger.layers)} layers")
            if not merger.layers:
                merger = None
        except Exception as e:
            print(f"[Rank {rank}] LoRA merge disabled: {e!r}")
            merger = None
    merge_checked = False

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = CFG["out_dir"]
    os.makedirs(dir_outputs, exist_ok=True)

    pacer = Pacer(shared, rank, n_workers, end_time, arc_test_set.queries)

    def bench_dfs(items, label):
        tokens = [t for _, t in items]
        torch.cuda.synchronize()
        t0 = time.time()
        steps0 = DFS_STEPS[0]
        with torch.inference_mode():
            res = inference_turbo_dfs(model, tokens, max_new_tokens, -np.log(CFG["min_prob"]), end_time, 600.0)
        torch.cuda.synchronize()
        dt = time.time() - t0
        steps = DFS_STEPS[0] - steps0
        beams = sorted((b, tuple(tk)) for _, bs in res for b, tk in [(round(x[0], 3), x[1]) for x in bs])
        print(f"[Rank {rank}] BENCH {label}: rows {len(items)} ctx {len(tokens[0])} time {dt:.1f}s "
              f"steps {steps} ms/step {1000 * dt / max(steps, 1):.1f} beams {len(beams)}")
        return dict(label=label, rows=len(items), ctx=len(tokens[0]), t=round(dt, 2), steps=steps), beams

    def run_task(key, plan):
        nonlocal model, merge_checked, merger, bench_pending

        start_time = time.time()
        deadline = min(start_time + plan["budget"], end_time)
        info = dict(task=key, rank=rank, pass_name=plan["pass_name"], level=plan["level"],
                    budget=round(plan["budget"], 1), est=round(plan["est"], 1), kappa=round(plan["kappa"], 3))

        torch.cuda.reset_peak_memory_stats()

        set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=plan["n_train_aug"], shfl_keys=True, seed=plan["train_seed"])
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        train_args_task = dict(train_args, seed=plan["lora_seed"])
        if plan["lora_seed"] != 42:
            torch.manual_seed(plan["lora_seed"])

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args_task),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        info["train_s"] = round(time.time() - start_time, 1)
        info["train_steps"] = int(stats.global_step)
        info["train_loss"] = round(float(stats.training_loss), 5)
        info["train_mem_mb"] = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] trained {key} ({plan['pass_name']}, level {plan['level']}, "
              f"{stats.global_step} steps) in {info['train_s']}s, loss {info['train_loss']}")

        torch.cuda.reset_peak_memory_stats()

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=plan["eval_seed"])
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        batches = make_batches(eval_ds, formatter, tokenizer, CFG["max_rows"], CFG["rows8_max_ctx"])

        bench = []
        if bench_pending and merger is not None and len(batches[0]) >= 8:
            bench_pending = False
            r, beams_unmerged = bench_dfs(batches[0], "unmerged")
            bench.append(r)

        # --- merge the adapter for decoding (verified once per worker) ---
        if merger is not None:
            try:
                probe_q = [eval_ds.get(batches[0][0][0], formatter)["input"]]
                probe_a = [formatter.fmt_reply([puzzle_ds_multi.queries[batches[0][0][0].split(".")[0]]["test"][0]["input"]])]
                if not merge_checked:
                    with torch.inference_mode():
                        ref = calc_scores(probe_q, probe_a, tokenizer, model)[0]
                merger.merge()
                if not merge_checked:
                    with torch.inference_mode():
                        got = calc_scores(probe_q, probe_a, tokenizer, model)[0]
                    ok = abs(got - ref) <= 0.03 * abs(ref) + 1.0
                    print(f"[Rank {rank}] merge self-test: unmerged NLL {ref:.4f} merged NLL {got:.4f} -> {'OK' if ok else 'MISMATCH'}")
                    info["merge_check"] = [round(ref, 4), round(got, 4), ok]
                    merge_checked = True
                    if not ok:
                        merger.restore()
                        merger = None
            except Exception as e:
                print(f"[Rank {rank}] merge failed, continuing unmerged: {e!r}")
                try:
                    merger.restore()
                except Exception:
                    pass
                merger = None

        if bench and merger is not None and merger.merged:
            r, beams_merged = bench_dfs(batches[0], "merged")
            bench.append(r)
            r["same_beams"] = [t for _, t in beams_merged] == [t for _, t in beams_unmerged]
            r, _ = bench_dfs(batches[0][:4], "merged-4rows")
            bench.append(r)
            info["bench"] = bench

        known_scores = {}
        sols_by_bk = defaultdict(list)
        n_views_done = defaultdict(int)
        dfs_log = []

        def decode_batch(items, max_score, suffix):
            subkeys = [s for s, _ in items]
            tokens = [t for _, t in items]
            window = CFG["window_per_view"] * len(items)
            t0 = time.time()
            steps0 = DFS_STEPS[0]
            try:
                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, min(deadline, end_time), window)
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if len(items) == 1:
                    print(f"[Rank {rank}] OOM on single view {subkeys[0]}, skipped")
                    return
                print(f"[Rank {rank}] OOM with {len(items)} rows, splitting")
                h = len(items) // 2
                decode_batch(items[:h], max_score, suffix)
                decode_batch(items[h:], max_score, suffix)
                return
            dt = time.time() - t0
            steps = DFS_STEPS[0] - steps0
            dfs_log.append((len(items), len(tokens[0]), round(dt, 1), steps, round(1000 * dt / max(steps, 1), 1)))
            for subkey in subkeys:
                n_views_done[subkey.split(".")[0]] += 1

            for subkey_id, scored_beams in dfs_result:

                subkey = subkeys[subkey_id]
                bk = subkey.split(".")[0]
                decoded_result = []

                for beam_score, toks in scored_beams:

                    array = formatter.convert_tokens_to_array(toks)
                    if array is None:
                        continue

                    solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                    grid_id = (bk, tuple(map(tuple, solution)))

                    if grid_id in known_scores:
                        augmented_scores = known_scores[grid_id]
                    else:
                        aug_dataset = ArcDataset(
                            keys=[bk],
                            queries={bk: puzzle_ds_multi.queries.get(bk)},
                            replies={bk: [solution.tolist()]},
                        )
                        aug_dataset = aug_dataset.augment(seed=stable_seed(bk))
                        aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                        aug_queries = []
                        aug_answers = []
                        for augmented_sample in aug_dataset.as_list(formatter):
                            aug_queries.append(augmented_sample["input"])
                            aug_answers.append(augmented_sample["reply"])
                        augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                        augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                        augmented_scores = augmented_scores1 + augmented_scores2
                        known_scores[grid_id] = augmented_scores

                    decoded_result.append({
                        "beam_score": beam_score,
                        "score_aug": augmented_scores,
                        "solution": solution,
                    })
                    sols_by_bk[bk].append(grid_id[1])

                if len(decoded_result):
                    with bz2.BZ2File(os.path.join(dir_outputs, subkey + suffix), "w") as f:
                        pickle.dump(decoded_result, f)

        timed_out = False
        with torch.inference_mode():

            for items in batches:
                if time.time() > deadline - 5 or time.time() > end_time - 30:
                    timed_out = True
                    break
                decode_batch(items, -np.log(CFG["min_prob"]), plan["suffix"])

            info["base_s"] = round(time.time() - start_time, 1)

            # --- adaptive extra views for outputs without a dominant candidate ---
            extra_done = 0
            if plan["extra_views"] and not timed_out:
                bks = sorted(puzzle_ds_multi.keys)
                todo = [bk for bk in bks if not output_confident(sols_by_bk, bk, n_views_done[bk])]
                if todo:
                    extra_ds = puzzle_ds_multi.change_keys(todo).augment(n=1, seed=plan["eval_seed"] + 1000)
                    extra_ds = extra_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                    extra_batches = make_batches(extra_ds, formatter, tokenizer, CFG["max_rows"], CFG["rows8_max_ctx"])
                    for items in extra_batches:
                        if time.time() > deadline - 5 or time.time() > end_time - 30:
                            break
                        decode_batch(items, -np.log(CFG["extra_min_prob"]), plan["suffix"] + ".x")
                        extra_done += len(items)
                info["extra_outputs"] = len(todo)
            info["extra_views"] = extra_done

        timed_out = timed_out or time.time() >= deadline - 2

        if merger is not None:
            merger.restore()

        # uncertainty summary for the tail pass
        unc = {}
        for bk in sorted(puzzle_ds_multi.keys):
            counts = defaultdict(int)
            for s in sols_by_bk.get(bk, []):
                counts[s] += 1
            top = sorted(counts.values(), reverse=True)
            unc[bk] = dict(n_unique=len(counts), c1=top[0] if top else 0,
                           c2=top[1] if len(top) > 1 else 0, views=n_views_done[bk])
        info["unc"] = unc
        info["dfs"] = dfs_log
        info["infer_mem_mb"] = torch.cuda.max_memory_allocated() // 1024**2
        info["elapsed"] = round(time.time() - start_time, 1)
        info["timed_out"] = timed_out
        tel(CFG["telemetry"], info)
        print(f"[Rank {rank}] finished {key} ({plan['pass_name']}) in {info['elapsed']}s "
              f"[train {info['train_s']}s, base views {info['base_s']}s, extra views {extra_done}, "
              f"budget {plan['budget']:.0f}s, dfs {dfs_log}]")
        return info

    while True:
        plan = pacer.next_task()
        if plan is None:
            break
        key = plan["task"]
        t0 = time.time()
        try:
            info = run_task(key, plan)
            pacer.done(plan, time.time() - t0, info)
        except Exception as e:
            print(f"[Rank {rank}] ERROR on {key}: {e!r}")
            traceback.print_exc()
            pacer.done(plan, time.time() - t0, None)
            try:
                if merger is not None:
                    merger.restore()
            except Exception as e2:
                print(f"[Rank {rank}] restore after error failed: {e2!r}; disabling merge")
                merger = None
            gc.collect()
            torch.cuda.empty_cache()

    print(f"[Rank {rank}] no more work")


Writing /kaggle/working/floor/arc_solver.py


In [18]:
%%writefile /kaggle/working/floor/starter.py
import os
import time
import json
import traceback
import torch
import argparse
import torch.multiprocessing as mp

N_WORKERS = int(os.getenv("ARC_N_WORKERS", "4"))
FLAG_DIR = os.getenv("ARC_FLAG_DIR", "/kaggle")


def local_worker(rank, end_time, shared, test_path):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"{FLAG_DIR}/worker{rank-1}"):
            time.sleep(5)

    # A raised exception would make mp.spawn terminate the other three workers.
    worker = None
    try:
        from arc_solver import worker
    except BaseException as e:
        print(f"[Rank {rank}] FATAL import: {e!r}")
        traceback.print_exc()
    finally:
        with open(f"{FLAG_DIR}/worker{rank}", "w") as f:
            f.write("Ok")

    print(f"[Rank {rank}] start!")

    if worker is not None:
        try:
            worker(rank, end_time, shared, N_WORKERS, test_path)
        except BaseException as e:
            print(f"[Rank {rank}] FATAL: {e!r}")
            traceback.print_exc()

    print(f"[Rank {rank}] done!")


def resolve_test_path(rerun_mode):
    if os.getenv("ARC_TEST_PATH"):
        return os.getenv("ARC_TEST_PATH")
    name = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    for root in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2", "/kaggle/input/arc-prize-2026-arc-agi-2"):
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return os.path.join("/kaggle/input/competitions/arc-prize-2026-arc-agi-2", name)


def main(worker_fn=local_worker):

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    from arc_sched import est_task_seconds, est_train_seconds, est_decode_seconds

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
    test_path = resolve_test_path(rerun_mode)

    with open(test_path, "r") as f:
        data = json.load(f)
    # APEX-5: pending filter - the floor only solves tasks the agent left unanswered
    _pf = os.getenv("ARC_PENDING_FILE", "")
    if _pf and os.path.exists(_pf):
        _pend = set(json.load(open(_pf)))
        data = {k: v for k, v in data.items() if k in _pend}
        print(f"[floor-starter] pending filter: {len(data)} tasks")

    if not rerun_mode:
        keys = os.getenv("ARC_COMMIT_TASKS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
        data = {k: data[k] for k in keys if k in data}

    est = {k: est_task_seconds(t) for k, t in data.items()}
    # cheapest (and most often solved) tasks first: whatever the time pressure on the
    # hidden set turns out to be, it can only squeeze the large low-yield tail
    order = sorted(data.keys(), key=lambda k: (est[k], k))

    budget = args.end_time - time.time()
    print(f"[starter] {len(order)} tasks, predicted baseline work {sum(est.values())/3600:.2f} GPU-h, "
          f"available {N_WORKERS * budget / 3600:.2f} GPU-h, test_path={test_path}")

    manager = mp.Manager()
    shared = dict(
        lock=manager.Lock(),
        order=manager.list(order),
        est=manager.dict(est),
        state=manager.dict(qi=0, tr_act=0.0, tr_exp=0.0, dec_act=0.0, dec_exp=0.0,
                           rem_tr0=float(sum(est_train_seconds(t) for t in data.values())),
                           rem_dec0=float(sum(est_decode_seconds(t) for t in data.values()))),
        busy=manager.dict(),
        unc=manager.dict(),
        tail_done=manager.dict(),
        failed=manager.dict(),
    )

    mp.spawn(worker_fn, args=(args.end_time, shared, test_path), nprocs=N_WORKERS, join=True)

    st = dict(shared["state"])
    print(f"[starter] finished: main queue reached {st['qi']}/{len(order)}, tail runs {len(shared['tail_done'])}, "
          f"kappa train {(1500 + st['tr_act']) / (1500 + st['tr_exp']):.3f} decode {(1500 + st['dec_act']) / (1500 + st['dec_exp']):.3f}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/floor/starter.py


In [19]:
%%writefile /kaggle/working/floor/floor_fills.py
"""APEX-5 residual floor: after the agent phase submit_main() kills the vLLM server in
its finally block, so all 4 L4s are free. If enough time remains inside the 12 h wall,
run the proven NVARC 4B TTT floor on ONLY the tasks the agent left unanswered (fallback
signature), then fill ONLY those slots. It never displaces a real agent grid and never
raises - if anything fails, the agent submission stands.

    python floor_fills.py
"""
import glob
import json
import os
import subprocess
import sys
import time

WORK = "/kaggle/working"
FLOOR = f"{WORK}/floor"
SUB = f"{WORK}/submission.json"


def one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def main():
    t0 = time.time()
    start_epoch = float(open(f"{WORK}/start_epoch.txt").read().strip())
    wall_end = start_epoch + 12 * 3600
    floor_end = wall_end - 32 * 60            # stop 32 min before the wall: save/upload margin
    t_left = floor_end - time.time()
    print(f"[floor] {t_left / 60:.0f} min to floor_end")
    if t_left < 70 * 60:
        print("[floor] insufficient residual time; agent submission stands")
        return
    if not os.path.exists(SUB):
        print("[floor] no agent submission found; nothing to fill")
        return

    ch = json.load(open(one("/kaggle/input/**/arc-agi_test_challenges.json")))
    sub = json.load(open(SUB))

    # pair-level pending: (tid, j) slots still holding the fallback input-copy
    pending_tasks = {}
    fill_slots = {}
    for tid, task in ch.items():
        tests = task["test"]
        att = sub.get(tid)
        if not isinstance(att, list) or len(att) < len(tests):
            pending_tasks[tid] = True
            fill_slots[tid] = list(range(len(tests)))
            continue
        js = []
        for j, x in enumerate(tests):
            a = att[j] if j < len(att) else {}
            if a.get("attempt_1") == x["input"] or a.get("attempt_2") == x["input"]:
                js.append(j)
        if js:
            pending_tasks[tid] = True
            fill_slots[tid] = js
    if not pending_tasks:
        print("[floor] agent answered everything; nothing to fill")
        return
    print(f"[floor] pending: {len(pending_tasks)} tasks, {sum(len(v) for v in fill_slots.values())} slots")
    json.dump(sorted(pending_tasks), open(f"{FLOOR}/pending.json", "w"))

    model_cfg = one("/kaggle/input/**/qwen3_4b_grids15_sft139/**/config.json")
    env = dict(os.environ)
    env.update({
        "PYTHONHASHSEED": "260618",
        "ARC_AUG_SEED_OFFSET": "260618",
        "UNSLOTH_DISABLE_STATISTICS": "1",
        "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
        "OMP_NUM_THREADS": "12",
        "ARC_N_WORKERS": "4",
        "ARC_MODEL_PATH": model_cfg.rsplit("/", 1)[0],
        "ARC_OUT_DIR": f"{FLOOR}/outs",
        "ARC_TELEMETRY": f"{FLOOR}/telemetry.jsonl",
        "ARC_MIN_TASK_S": "300",
        "ARC_ORDER": "sjf",
        "ARC_FAIR_SHARE": "1",
        "ARC_PENDING_FILE": f"{FLOOR}/pending.json",
    })
    end_t = floor_end - 3 * 60
    print(f"[floor] 4B TTT on {len(pending_tasks)} tasks until +{(end_t - time.time()) / 60:.0f} min")
    try:
        subprocess.run([sys.executable, "starter.py", "--end-time", str(end_t)],
                       cwd=FLOOR, env=env,
                       timeout=max(300.0, end_t - time.time() + 150))
    except BaseException as e:
        print("[floor] runner ended:", repr(e))

    # ---- assemble the floor submission (same algorithm as the proven Phase-A cell) ----
    floor_grids = {}
    try:
        sys.path.insert(0, FLOOR)
        import arc_envfix  # noqa: F401  (py3.12 utility-copy guard, harmless on py3.11)
        os.environ["ARC_OUT_DIR"] = f"{FLOOR}/outs"
        import arc_loader
        import arc_decoder
        data_dir = one("/kaggle/input/**/arc-agi_test_challenges.json").rsplit("/", 1)[0]
        data = arc_loader.ArcDataset.from_file(f"{data_dir}/arc-agi_test_challenges.json")
        decoder = arc_decoder.ArcDecoder(data.split_multi_replies(), n_guesses=2)
        if os.path.isdir(f"{FLOOR}/outs"):
            decoder.load_decoded_results(f"{FLOOR}/outs")
        floor_sub = data.get_submission(decoder.run_selection_algo())
        for tid in pending_tasks:
            for j, a in enumerate(floor_sub.get(tid, [])):
                g1, g2 = a.get("attempt_1"), a.get("attempt_2")
                x = ch[tid]["test"][j]["input"]
                # placeholder signature is input-copy; a real identity answer with a
                # different att2 is accepted (g2 != x proves the floor produced grids)
                if g1 is None:
                    continue
                if g1 != x:
                    floor_grids.setdefault(tid, {})[j] = (g1, g2 if g2 != x else None)
                elif g2 is not None and g2 != x:
                    floor_grids.setdefault(tid, {})[j] = (g1, g2)
    except BaseException as e:
        print("[floor] assembly failed (agent submission stands):", repr(e))

    if not floor_grids:
        print(f"[floor] no floor results to merge; done {(time.time() - t0) / 60:.0f} min")
        return

    # ---- fills-only merge, pair-level, never displace ----
    n_fill = 0
    for tid, slots in fill_slots.items():
        for j in slots:
            fg = floor_grids.get(tid, {}).get(j)
            if not fg:
                continue
            x = ch[tid]["test"][j]["input"]
            att = sub[tid]
            while len(att) <= j:
                att.append({"attempt_1": x, "attempt_2": x})
            if att[j].get("attempt_1") == x:
                att[j]["attempt_1"] = fg[0]
                if att[j].get("attempt_2") == x and fg[1]:
                    att[j]["attempt_2"] = fg[1]
            elif att[j].get("attempt_2") == x and fg[1]:
                att[j]["attempt_2"] = fg[1]
            else:
                continue
            n_fill += 1
    tmp = SUB + ".tmp"
    with open(tmp, "w") as f:
        json.dump(sub, f)
    os.replace(tmp, SUB)
    print(f"[floor] filled {n_fill} slots in {(time.time() - t0) / 60:.0f} min")


if __name__ == "__main__":
    try:
        main()
    except BaseException:
        import traceback
        print("[floor] FATAL (agent submission stands):")
        traceback.print_exc()


Writing /kaggle/working/floor/floor_fills.py


In [20]:
import json, os, subprocess, sys, time
from pathlib import Path
START = float(Path("/kaggle/working/start_epoch.txt").read_text())
cfg = json.loads(r"""{"model_glob": "/kaggle/input/**/qwen3-8-27b-fp8-repacked-v1/**/config.json", "overlay": "/kaggle/input/**/taaf-duck-qwen38-serving-v1/serving_wheels/*.whl", "parsers": ["--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3"], "server_ladder": [["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--language-model-only", "--speculative-config", "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--speculative-config", "{\"method\": \"qwen3_5_mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8", "--speculative-config", "{\"method\": \"mtp\", \"num_speculative_tokens\": 2}"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}", "--kv-cache-dtype", "fp8"], ["--reasoning-config", "{\"reasoning_start_str\": \"<think>\", \"reasoning_end_str\": \"</think>\"}"], []], "max_model_len": 81920, "max_num_seqs": 16, "mode": "direct", "samples": 2, "concurrency": 10, "min_start_s": 600, "template_kwargs": {"reasoning_effort": "xhigh"}, "temperature": 1.0, "top_p": 0.95, "top_k": 20, "thinking_token_budget": 40960, "max_tokens": 49152, "rescue_tokens": 8192, "tok_per_s": 140}""")
cfg["deadline_epoch"] = START + 12 * 3600 - 35 * 60        # 35 min reserve inside the 12 h limit
cfg["deadline_epoch"] -= 6000  # APEX-5: pull the agent in by 100 min -> residual floor window
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
if not RERUN:
    # A version is only submittable once a run has produced submission.json. The save run
    # therefore writes the fallback and stops here (a minute of quota); the agent itself
    # runs only in the competition rerun, which does not bill quota.
    print("not a competition rerun: fallback submission only, agent skipped")
else:
    try:
        rc = subprocess.run([sys.executable, "/kaggle/working/agent/submit_main.py", json.dumps(cfg)],
                            timeout=max(60, cfg["deadline_epoch"] + 15 * 60 - time.time())).returncode
        print("submit_main exit code", rc)
    except Exception as e:
        print("submit_main did not finish cleanly:", repr(e))
    # APEX-5 residual floor: submit_main's finally killed the server; 4 GPUs are free.
    # Fills ONLY slots the agent left as fallback copies; never displaces; never raises.
    try:
        rc2 = subprocess.run([sys.executable, "/kaggle/working/floor/floor_fills.py"],
                             timeout=max(300.0, START + 12 * 3600 - 31 * 60 - time.time()))
        print("floor_fills exit code", rc2.returncode)
    except Exception as e2:
        print("floor phase skipped:", repr(e2))


not a competition rerun: fallback submission only, agent skipped


In [21]:
# Last line of defence: a malformed file is replaced by the fallback.
import json, glob
try:
    ch = json.load(open(glob.glob("/kaggle/input/**/arc-agi_test_challenges.json", recursive=True)[0]))
    sub = json.load(open("/kaggle/working/submission.json"))
    def ok_grid(g):
        return (isinstance(g, list) and g and all(isinstance(r, list) and r and len(r) == len(g[0]) for r in g)
                and all(isinstance(v, int) and 0 <= v <= 9 for r in g for v in r))
    bad = [t for t, task in ch.items()
           if t not in sub or len(sub[t]) != len(task["test"])
           or not all(ok_grid(r.get("attempt_1")) and ok_grid(r.get("attempt_2")) for r in sub[t])]
    if bad:
        print("INVALID rows for", len(bad), "tasks -> fallback for those")
        for t in bad:
            sub[t] = [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in ch[t]["test"]]
        json.dump(sub, open("/kaggle/working/submission.json", "w"))
    changed = sum(r["attempt_1"] != x["input"] for t in ch for r, x in zip(sub[t], ch[t]["test"]))
    print(f"submission.json valid: {len(sub)} tasks, {changed} test outputs answered by the agent")
except Exception as e:
    print("validation error:", repr(e))


submission.json valid: 240 tasks, 0 test outputs answered by the agent
